# 21 — Spatial evaluation, error analysis and relative colour contract (V3)

Evaluates the **accepted** notebook-20 selection (`backend/rerun-spatial-model-016-002`) from its saved full-population arrays.
No training, no reselection, no serving, no frontend/API and **no 2026 data**.

- Inputs are verified against the manager pins (`workflow/evaluation/input-pins-v3.json`) and the accepted 20/19 manifests before any value is read.
- Label = SELECTED+LINKED retained MOT recorded count per DOH segment × local day / 6 h block. Zero ≠ no physical crash.
- Colour density = expected recorded count / segment km for the same horizon; cutoffs = P50/P80/P95 of the selected policy's 2022–23 OOF cells,
  separately for daily and 6 h. *Base* means lower **relative** predicted density — never "safe", never an absolute or per-vehicle probability.
- The metric-floor audit was registered after preliminary daily scores were seen: a disclosed sensitivity, not a selection gate.
- All logic lives in the four `# FUNCTIONS` cells; the runner `backend/scripts/run_spatial_evaluation.py` only sets roots and executes.


In [ ]:
# PARAMETERS: roots come from the runner environment (ROADAWARE_BACKEND_ROOT / ROADAWARE_V3_OUTPUT_ROOT). Outputs are only accepted under a NEW
# backend/rerun-spatial-eval-* root, so an interactive run must set ROADAWARE_V3_OUTPUT_ROOT explicitly.
import os
from pathlib import Path


def _default_backend(c):
    for p in (c, *c.parents):
        if (p / 'config' / 'spatial_evaluation_v3.json').is_file():
            return p
        if (p / 'backend' / 'config' / 'spatial_evaluation_v3.json').is_file():
            return p / 'backend'
    return c


BACKEND = Path(os.environ.get('ROADAWARE_BACKEND_ROOT') or _default_backend(Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.environ.get('ROADAWARE_V3_OUTPUT_ROOT') or BACKEND).resolve()
CONFIG_PATH = BACKEND / 'config' / 'spatial_evaluation_v3.json'
print('backend', BACKEND, '| output root', OUTPUT_ROOT)


In [ ]:
# FUNCTIONS (1/4): strict JSON/hash IO, manager pins, verification of every consumed accepted-20 / accepted-19 file against the pins AND its
# producer manifest, hash/shape/dtype-checked array loading (no pickle, no model deserialisation), output and protected-path guards.
import datetime as dt
import hashlib
import json
import math
import os
import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


class EvaluationError(Exception):
    pass


def utcnow():
    return dt.datetime.now(dt.timezone.utc).isoformat()


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


def sha256_bytes(b):
    return hashlib.sha256(bytes(b)).hexdigest()


def _no_constant(x):
    raise EvaluationError('non-strict JSON constant ' + x)


def read_strict_json(p):
    return json.loads(Path(p).read_text('utf-8'), parse_constant=_no_constant)


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return jsonable(o.tolist())
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (float, np.floating)):
        f = float(o)
        return f if math.isfinite(f) else None
    if isinstance(o, Path):
        return str(o)
    return o


def json_bytes(obj):
    return (json.dumps(jsonable(obj), indent=2, ensure_ascii=False, allow_nan=False) + '\n').encode('utf-8')


def csv_bytes(df):
    return df.to_csv(index=False, lineterminator='\n').encode('utf-8')


def atomic_write_bytes(p, data):
    """Exclusive write: never overwrites, never follows an existing path."""
    p = Path(p)
    if os.path.lexists(p):
        raise EvaluationError('refusing to overwrite ' + str(p))
    tmp = p.with_name('.' + p.name + '.tmp-' + str(os.getpid()))
    with open(tmp, 'xb') as f:
        f.write(bytes(data))
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, p)
    return sha256_file(p)


def require_modules(pins):
    got = {}
    for name, want in pins.items():
        mod = __import__(name)
        got[name] = getattr(mod, '__version__', None)
        if want is not None and got[name] != want:
            raise EvaluationError(f'{name} {got[name]} != pinned {want}')
    return got


def environment(versions):
    return {'python': platform.python_version(), 'platform': platform.platform(), 'dependencies': versions}


def load_pins(backend, cfg):
    p = Path(backend) / cfg['pins']['path']
    pins = read_strict_json(p)
    if pins.get('kind') != cfg['pins']['kind']:
        raise EvaluationError('unexpected pins kind ' + str(pins.get('kind')))
    if pins.get('accepted_run') != cfg['pins']['repo_prefix'] + cfg['inputs']['accepted_root']:
        raise EvaluationError('pins accepted_run does not name the configured accepted root')
    return pins, sha256_file(p)


def code_sources(nb_path):
    nb = read_strict_json(nb_path)
    return [(''.join(c['source']) if isinstance(c['source'], list) else c['source'], c) for c in nb['cells'] if c.get('cell_type') == 'code']


def verify_inputs(backend, cfg, pins):
    """Every manager-pinned file hash, then every consumed 20 report/data file against the accepted 20 manifest and every consumed 19 data
    file against the accepted 19 manifest. Returns (input rows, parsed documents)."""
    backend, I = Path(backend), cfg['inputs']
    repo, pre = backend.parent, cfg['pins']['repo_prefix']
    rows = []
    for key, want in sorted(pins['files'].items()):
        got = sha256_file(repo / key)
        rows.append({'input': key, 'source': 'manager_pin', 'sha256': got, 'expected': want, 'ok': got == want})
    root = backend / I['accepted_root']
    rep, dat = root / I['report_dir20'], root / I['data_dir20']
    m20 = read_strict_json(rep / 'spatial_experiments_manifest.json')
    m19 = read_strict_json(backend / I['manifest19'])
    rows.append({'input': 'manifest20.status', 'source': 'accepted20', 'sha256': m20.get('status'), 'expected': 'complete', 'ok': m20.get('status') == 'complete'})
    rows.append({'input': 'manifest19.status', 'source': 'accepted19', 'sha256': m19.get('status'), 'expected': 'complete', 'ok': m19.get('status') == 'complete'})
    rows.append({'input': 'manifest20.config_sha256', 'source': 'accepted20', 'sha256': m20.get('config_sha256'),
                 'expected': pins['files'][pre + I['config20']], 'ok': m20.get('config_sha256') == pins['files'][pre + I['config20']]})
    for name in I['reports20_used']:
        got = sha256_file(rep / name)
        rows.append({'input': I['accepted_root'] + '/' + I['report_dir20'] + '/' + name, 'source': 'manifest20.report_files', 'sha256': got,
                     'expected': m20['report_files'].get(name), 'ok': got == m20['report_files'].get(name)})
    got = sha256_file(dat / 'arrays_index.json')
    rows.append({'input': 'arrays_index.json', 'source': 'manifest20.data_files_gitignored', 'sha256': got,
                 'expected': m20['data_files_gitignored'].get('arrays_index.json'), 'ok': got == m20['data_files_gitignored'].get('arrays_index.json')})
    for name in I['data19_used']:
        got = sha256_file(backend / I['prepared_data_dir'] / name)
        rows.append({'input': I['prepared_data_dir'] + '/' + name, 'source': 'manifest19.data_files_gitignored', 'sha256': got,
                     'expected': m19['data_files_gitignored'].get(name), 'ok': got == m19['data_files_gitignored'].get(name)})
    src = code_sources(backend / I['notebook20'])
    unexecuted = all(not c.get('outputs') and c.get('execution_count') is None for _, c in src)
    rows.append({'input': I['notebook20'], 'source': 'source notebook unexecuted', 'sha256': None, 'expected': None, 'ok': unexecuted})
    exe = code_sources(root / I['executed_notebook20'])
    same = [s for s, _ in src] == [s for s, _ in exe]
    rows.append({'input': I['accepted_root'] + '/' + I['executed_notebook20'], 'source': 'executed code cells == canonical source cells',
                 'sha256': sha256_file(root / I['executed_notebook20']), 'expected': None, 'ok': same})
    lock = read_strict_json(rep / 'experiment_lock.json')
    sel = read_strict_json(rep / 'selection_frozen.json')
    idx = read_strict_json(dat / 'arrays_index.json')
    lock_sha = pins['files'][pre + I['accepted_root'] + '/' + I['report_dir20'] + '/experiment_lock.json']
    for what, v in (('manifest20', m20.get('experiment_lock_sha256')), ('selection_frozen', sel.get('experiment_lock_sha256')), ('arrays_index', idx.get('experiment_lock_sha256'))):
        rows.append({'input': what + '.experiment_lock_sha256', 'source': 'lock consistency', 'sha256': v, 'expected': lock_sha, 'ok': v == lock_sha})
    docs = {'manifest20': m20, 'manifest19': m19, 'lock': lock, 'selection': sel, 'index': idx,
            'config20': read_strict_json(backend / I['config20']), 'root': root, 'report_dir20': rep, 'data_dir20': dat}
    return pd.DataFrame(rows), docs


def array_rel(scope, meta):
    return 'arrays/' + ('transfer/' if scope == 'transfer' else '') + meta['file']


def load_array(docs, unit, scope, name):
    """Memory-mapped .npy (allow_pickle=False) after sha256 == arrays_index AND == accepted manifest membership, then dtype/shape/finite checks."""
    meta = docs['index']['units'][unit][scope][name]
    rel = array_rel(scope, meta)
    p = docs['data_dir20'] / rel
    got = sha256_file(p)
    if got != meta['sha256'] or got != docs['manifest20']['data_files_gitignored'].get(rel):
        raise EvaluationError(f'array hash mismatch {rel}')
    a = np.load(p, mmap_mode='r', allow_pickle=False)
    if str(a.dtype) != meta['dtype'] or list(a.shape) != list(meta['shape']):
        raise EvaluationError(f'array dtype/shape mismatch {rel}: {a.dtype} {a.shape}')
    return a, {'unit': unit, 'scope': scope, 'name': name, 'file': rel, 'sha256': got, 'dtype': str(a.dtype), 'shape': list(a.shape)}


def method_components(method):
    """A directly stored method is used as stored; only a missing '<cfg>::routed_context' is derived from mask/context/core."""
    if method.endswith('::routed_context'):
        cid = method.split('::')[0]
        return [cid + '::context', cid + '::core', 'context_mask']
    return [method]


def load_method(docs, unit, scope, method):
    """Returns (pred(lo, hi) -> float64 ndarray, array metas). Stored-first; derivation where(mask == 1, context, core) otherwise."""
    have = docs['index']['units'][unit][scope]
    if method in have:
        a, m = load_array(docs, unit, scope, method)
        return (lambda lo, hi: np.asarray(a[lo:hi], dtype=np.float64)), [m]
    names = method_components(method)
    if len(names) != 3 or not all(n in have for n in names):
        raise EvaluationError(f'{unit}/{scope}: method {method} neither stored nor derivable')
    (ctx, mc), (core, mo), (mask, mm) = (load_array(docs, unit, scope, n) for n in names)
    return (lambda lo, hi: np.where(np.asarray(mask[lo:hi]) == 1, np.asarray(ctx[lo:hi], np.float64), np.asarray(core[lo:hi], np.float64))), [mc, mo, mm]


def _parts(path):
    p = Path(os.path.abspath(path))
    return [p, *p.parents]


def overlaps(a, b):
    a, b = Path(os.path.abspath(a)), Path(os.path.abspath(b))
    ra, rb = a.resolve(), b.resolve()
    return a == b or b in a.parents or a in b.parents or ra == rb or rb in ra.parents or ra in rb.parents


def output_paths(out_root, cfg):
    o, out = cfg['outputs'], Path(out_root)
    rd = out / o['report_dir']
    return {'root': out, 'report_dir': rd, 'figure_dir': rd / o['figure_subdir'], 'data_dir': out / o['data_dir']}


def guarded_paths(backend, cfg):
    backend, I = Path(backend), cfg['inputs']
    g = [backend / p for p in cfg['protected_paths']]
    g += [backend / I[k] for k in ('notebook20', 'config20', 'notebook19', 'config19', 'manifest19', 'prepared_data_dir', 'province_names') if k in I]
    g += [backend / I['accepted_root'], backend / cfg['pins']['path'], backend / cfg['notebook'], backend / 'config' / 'spatial_evaluation_v3.json']
    return g


def output_problems(paths, backend, cfg):
    """Pure pre-write guard: NEW rerun-spatial-eval-* root directly inside backend/, no symlinked component, no existing report/data dir,
    and no overlap (either direction, also after resolving aliases) with any protected or input path."""
    backend_abs = Path(os.path.abspath(backend))
    backend, root = Path(backend).resolve(), Path(os.path.abspath(paths['root']))
    probs = []
    if not root.name.startswith(cfg['outputs']['root_prefix']) or root.name == cfg['outputs']['root_prefix']:
        probs.append('output root name must start with ' + cfg['outputs']['root_prefix'])
    if root.parent.resolve() != backend:
        probs.append('output root must sit directly inside ' + str(backend))
    for k in ('report_dir', 'data_dir', 'figure_dir'):
        w = Path(os.path.abspath(paths[k]))
        if os.path.lexists(w):
            probs.append(k + ' already exists: ' + str(w))
        for a in _parts(w):
            if a in (backend, backend_abs) or not (backend in a.parents or backend_abs in a.parents):
                break
            if os.path.islink(a):
                probs.append('symlinked output component ' + str(a))
        for g in guarded_paths(backend, cfg):
            if overlaps(w, g):
                probs.append(f'{k} {w} overlaps guarded path {g}')
    return probs


def province_name_map(df):
    """{str(int(province_code)): {'name_en', 'name_th'}} for report labels only; codes must be unique and finite."""
    if not {'province_code', 'name_en', 'name_th'} <= set(df.columns):
        raise EvaluationError('province names need province_code, name_en, name_th')
    code = pd.to_numeric(df['province_code'], errors='coerce')
    if code.isna().any() or code.duplicated().any():
        raise EvaluationError('province_code must be unique and numeric')
    return {str(int(c)): {'name_en': str(e), 'name_th': str(t)} for c, e, t in zip(code, df['name_en'], df['name_th'])}


def hash_paths(backend, rels):
    out = {}
    for rel in rels:
        p = Path(backend) / rel
        files = [p] if p.is_file() else sorted(q for q in p.rglob('*') if q.is_file()) if p.is_dir() else []
        for f in files:
            out[os.path.relpath(f, backend).replace(os.sep, '/')] = sha256_file(f)
    return out


In [ ]:
# FUNCTIONS (2/4): exact, batch-mergeable metric accumulators grouped by integer codes (np.bincount sums), the same Poisson deviance as
# notebook 20 (prediction floored ONLY inside the metric), population axes -> per-cell date/segment/block indices, group dimensions,
# reconciliation against the accepted 20 tables, the registered metric-floor sensitivity audit (NOT a selection gate), the streamed
# per-scope evaluation pass (uses the FUNCTIONS 3 colour/calibration functions at call time) and the province population reconciliation.
FIELDS = ('n', 'dv', 'ae', 'se', 'pred', 'actual', 'zero', 'nonzero', 'dv_zero', 'dv_nonzero', 'pred_zero', 'pred_nonzero', 'km',
          'floor_cells', 'floor_pos', 'floor_dv', 'floor_actual')
WEEKDAYS = ('Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun')


def poisson_deviance_terms(y, mu, floor):
    """2*(y*log(y/m) - (y - m)) with m = max(mu, floor); y*log(y/m) := 0 for y == 0. Inputs must be finite, y >= 0, mu >= 0."""
    y, mu = np.asarray(y, np.float64), np.asarray(mu, np.float64)
    if not (np.isfinite(y).all() and np.isfinite(mu).all() and (y >= 0).all() and (mu >= 0).all()):
        raise EvaluationError('deviance inputs must be finite and non-negative')
    m = np.maximum(mu, float(floor))
    t = np.zeros_like(y)
    pos = y > 0
    t[pos] = y[pos] * np.log(y[pos] / m[pos])
    return 2.0 * (t - (y - m))


def new_acc(g):
    return np.zeros((int(g), len(FIELDS)), np.float64)


def acc_add(acc, y, mu, km, floor, codes, dv=None):
    """Adds one batch. km = segment length (km) per cell, so 'km' sums to segment-km x periods (the exposure denominator).
    dv may be passed when already computed for the same (y, mu, floor)."""
    dv = poisson_deviance_terms(y, mu, floor) if dv is None else dv
    if codes.size and (codes.min() < 0 or codes.max() >= acc.shape[0]):
        raise EvaluationError('group code outside the accumulator')
    z, fl = y == 0, mu < float(floor)
    cols = (None, dv, np.abs(mu - y), np.square(mu - y), mu, y, z, ~z, dv * z, dv * ~z, mu * z, mu * ~z, km, fl, fl & ~z, dv * fl, y * fl)
    g = acc.shape[0]
    for j, w in enumerate(cols):
        acc[:, j] += np.bincount(codes, weights=None if w is None else np.asarray(w, np.float64), minlength=g)[:g]
    return dv


def acc_metrics(r):
    r = dict(zip(FIELDS, (float(v) for v in r)))
    n, a = r['n'], r['actual']
    div = lambda x, d: x / d if d > 0 else None
    return {'n': int(n), 'mean_poisson_deviance': div(r['dv'], n), 'mae': div(r['ae'], n), 'rmse': math.sqrt(r['se'] / n) if n > 0 else None,
            'bias': div(r['pred'] - a, n), 'wape': div(r['ae'], a), 'wape_status': 'ok' if a > 0 else 'undefined_zero_actual',
            'total_pred': r['pred'], 'total_actual': a, 'pred_over_actual': div(r['pred'], a), 'zero_cells': int(r['zero']), 'nonzero_cells': int(r['nonzero']),
            'mean_deviance_zero_cells': div(r['dv_zero'], r['zero']), 'mean_deviance_nonzero_cells': div(r['dv_nonzero'], r['nonzero']),
            'mean_pred_zero_cells': div(r['pred_zero'], r['zero']), 'mean_pred_nonzero_cells': div(r['pred_nonzero'], r['nonzero']),
            'deviance_sum': r['dv'], 'exposure_segment_km_periods': r['km'], 'observed_records_per_km_period': div(a, r['km']),
            'predicted_records_per_km_period': div(r['pred'], r['km']), 'metric_floor_cells': int(r['floor_cells']),
            'metric_floor_actual_positive_cells': int(r['floor_pos']), 'metric_floor_actual_records': r['floor_actual'], 'metric_floor_deviance_sum': r['floor_dv']}


def axes_frame(axj, segs, dates, cfg):
    """Segment attributes in population-axis order and date attributes for the axis dates (no date after max_date_read is retained)."""
    S = segs.set_index('segment_id').reindex(axj['segments'])
    if S['geom_length_m'].isna().any() or not (np.isfinite(S['geom_length_m']).all() and (S['geom_length_m'] > 0).all()):
        raise EvaluationError('every axis segment needs a finite positive geom_length_m')
    D = dates[dates['target_date'] <= cfg['time']['max_date_read']].set_index('target_date').reindex(axj['dates'])
    if D['day_of_week'].isna().any():
        raise EvaluationError('date features missing for some axis dates')
    return S, D


def season_of(D):
    out = np.full(len(D), 'unknown', dtype=object)
    for s in ('cool', 'wet', 'hot'):
        out[D['season_' + s].to_numpy(float) == 1] = s
    return out


def festival_of(D):
    out = np.full(len(D), 'none', dtype=object)
    for fest in ('new_year', 'songkran'):
        for suffix, lab in (('post_4_7', 'post_4_7'), ('post_1_3', 'post_1_3'), ('pre_4_7', 'pre_4_7'), ('pre_1_3', 'pre_1_3'), ('in', 'in')):
            out[D['fest_' + fest + '_' + suffix].to_numpy(float) == 1] = fest + '_' + lab
    return out


def encode(values):
    labels, codes = np.unique(np.asarray(values, dtype=object).astype(str), return_inverse=True)
    return codes.astype(np.int64), [str(x) for x in labels]


def province_labels(S):
    """Segment province label (segment midpoint, current geometry): str(int(code)), or 'unknown' when the code is missing."""
    return S['province_code'].map(lambda v: 'unknown' if pd.isna(v) else str(int(float(v)))).to_numpy(object)


def build_dimensions(S, D, unit, cfg, period_ids):
    """Each dimension = (axis, per-axis codes, labels). axis in date/segment/block/mask/y/period. Unknown province is its own group."""
    prov = province_labels(S)
    dims = {'pooled': ('const', None, ['all']), 'period': ('period', None, list(period_ids)),
            'weekday': ('date', D['day_of_week'].to_numpy(int), list(WEEKDAYS)),
            'month': ('date', D['month'].to_numpy(int) - 1, [str(m) for m in range(1, 13)])}
    for name, vals in (('season', season_of(D)), ('festival', festival_of(D))):
        c, lab = encode(vals)
        dims[name] = ('date', c, lab)
    blocks = cfg['time']['blocks_6h'] if unit == '6h' else ['all_day']
    dims['time_block'] = ('block', None, list(blocks))
    for name, vals in (('province', prov), ('route_code', S['route_code'].astype(str).to_numpy(object))):
        c, lab = encode(vals)
        dims[name] = ('segment', c, lab)
    dims['context'] = ('mask', None, ['fallback_core', 'context_available'])
    dims['actual_zero'] = ('y', None, ['zero', 'nonzero'])
    return dims


def cell_index(lo, hi, n_seg, n_blk):
    r = np.arange(lo, hi, dtype=np.int64)
    return r // (n_seg * n_blk), (r // n_blk) % n_seg, r % n_blk


def dim_codes(dim, d, s, b, mask, y, period_codes):
    axis, per, _ = dim
    if axis == 'const':
        return np.zeros(len(d), np.int64)
    if axis == 'date':
        return per[d]
    if axis == 'segment':
        return per[s]
    if axis == 'block':
        return b
    if axis == 'mask':
        return (np.asarray(mask) == 1).astype(np.int64)
    if axis == 'y':
        return (y > 0).astype(np.int64)
    if axis == 'period':
        return period_codes
    raise EvaluationError('unknown axis ' + axis)


def period_codes_for(periods, lo, hi):
    out = np.full(hi - lo, -1, np.int64)
    r = np.arange(lo, hi)
    for i, p in enumerate(periods):
        out[(r >= p['row_offset']) & (r < p['row_offset'] + p['rows'])] = i
    if (out < 0).any():
        raise EvaluationError('rows outside the declared periods')
    return out


def rows_from_acc(acc, labels, **extra):
    return [dict(extra, group=labels[i], **acc_metrics(acc[i])) for i in range(acc.shape[0])]


def relative_reduction(base_dv, cand_dv):
    return None if base_dv is None or base_dv <= 0 or cand_dv is None else 1.0 - cand_dv / base_dv


def reconcile(mine, ref, keys, rtol, atol):
    """Problems list comparing recomputed metrics with the accepted table/JSON values (None/NaN must agree)."""
    probs = []
    for k in keys:
        a, b = mine.get(k), ref.get(k)
        isnan = lambda v: v is None or (isinstance(v, (float, np.floating)) and math.isnan(v))
        if isnan(a) or isnan(b):
            if not (isnan(a) and isnan(b)):
                probs.append(f'{k}: {a} vs {b}')
        elif not math.isclose(float(a), float(b), rel_tol=rtol, abs_tol=atol):
            probs.append(f'{k}: {a!r} vs {b!r}')
    return probs


def gate_status(sel_unit, pooled_base, pooled_cand, fold_base, fold_cand, gates):
    """Re-derives the locked gates from recomputed metrics (identical rules to the lock); used only to confirm, never to reselect."""
    rdr = relative_reduction(pooled_base['mean_poisson_deviance'], pooled_cand['mean_poisson_deviance'])
    mae_ratio = pooled_cand['mae'] / pooled_base['mae']
    wins = {f: fold_cand[f]['mean_poisson_deviance'] < fold_base[f]['mean_poisson_deviance'] for f in fold_base}
    g = {'deviance_reduction': rdr >= gates['min_relative_deviance_reduction'], 'mae_ratio': mae_ratio <= gates['max_mae_ratio'],
         'fold_wins': sum(wins.values()) >= gates['min_fold_wins']}
    return {'relative_deviance_reduction': rdr, 'mae_ratio': mae_ratio, 'fold_wins': wins, 'n_fold_wins': sum(wins.values()), 'gates': g,
            'all_pass': all(g.values()), 'frozen_recommended': sel_unit['recommended'], 'frozen_selected': sel_unit['selected_method'],
            'baseline_retained': sel_unit['recommended'] != 'candidate',
            'consistent_with_frozen': all(g.values()) == (sel_unit['recommended'] == 'candidate') and g == sel_unit['gates']}


FLOOR_GROUPS = ['both_positive_mean', 'baseline_floor_only', 'candidate_floor_only', 'both_floor']


def floor_codes(p_base, p_cand, floor):
    return (p_base < floor).astype(np.int64) + 2 * (p_cand < floor).astype(np.int64)


def floor_audit_rows(acc_b, acc_c, unit, scope):
    """Registered post-hoc sensitivity (after preliminary daily scores): how much of the pooled deviance gain comes from cells where the
    registered baseline predicts below the metric floor. Reported as a limitation; selection, baseline and metric stay locked."""
    tb, tc = acc_b[:, FIELDS.index('dv')].sum(), acc_c[:, FIELDS.index('dv')].sum()
    gain = tb - tc
    rows = []
    for i, g in enumerate(FLOOR_GROUPS):
        mb, mc = acc_metrics(acc_b[i]), acc_metrics(acc_c[i])
        gi = mb['deviance_sum'] - mc['deviance_sum']
        rows.append({'unit': unit, 'scope': scope, 'floor_group': g, 'cells': mb['n'], 'actual_positive_cells': mb['nonzero_cells'],
                     'actual_records': mb['total_actual'], 'baseline_deviance_sum': mb['deviance_sum'], 'candidate_deviance_sum': mc['deviance_sum'],
                     'deviance_gain': gi, 'share_of_total_gain': gi / gain if gain else None,
                     'relative_deviance_reduction_within_group': relative_reduction(mb['mean_poisson_deviance'], mc['mean_poisson_deviance'])})
    bf = sum(r['deviance_gain'] for r in rows if r['floor_group'] in ('baseline_floor_only', 'both_floor'))
    comp = [r for r in rows if r['floor_group'] in ('both_positive_mean', 'candidate_floor_only')]
    cb, cc = sum(r['baseline_deviance_sum'] for r in comp), sum(r['candidate_deviance_sum'] for r in comp)
    summary = {'unit': unit, 'scope': scope, 'total_gain': gain, 'baseline_floor_gain_share': bf / gain if gain else None,
               'complement_relative_deviance_reduction': 1.0 - cc / cb if cb > 0 else None,
               'pooled_relative_deviance_reduction': 1.0 - tc / tb if tb > 0 else None}
    return rows, summary


def scope_dims(S, D, unit, cfg, periods):
    """Group dimensions of one unit x scope plus the colour / metric-floor / period x colour groups (colour names from FUNCTIONS 3)."""
    dims = build_dimensions(S, D, unit, cfg, [p['id'] for p in periods])
    labs = list(COLOUR_LABELS) + [GREY]
    extra = {'colour': labs, 'floor': list(FLOOR_GROUPS), 'period_colour': [p['id'] + '|' + c for p in periods for c in labs]}
    return dims, extra


def evaluate_scope(unit, scope, ctx):
    """One bounded streamed pass over one unit x scope for every role in ctx['methods'] (must include 'selected' and 'baseline'):
    full-population metrics by every group dimension, colour band (locked contract), calibration bin, metric-floor group, period x colour.
    ctx: S, D (axis frames), cfg, periods, methods {role: method}, pred {role: pred(lo, hi)}, y, mask, n_rows, n_seg, n_blk, date0,
    km (per segment), contract, cal_edges {role: edges}, floor, batch_rows. Returns (metrics DataFrame, accumulators)."""
    periods, methods, floor = ctx['periods'], ctx['methods'], ctx['floor']
    dims, extra = scope_dims(ctx['S'], ctx['D'], unit, ctx['cfg'], periods)
    cal = ctx['cal_edges']
    cal_labels = {r: [f'[{a!r}, {b!r})' for a, b in zip(cal[r][:-1], cal[r][1:])] for r in methods}
    sizes = dict({k: len(v[2]) for k, v in dims.items()}, **{k: len(v) for k, v in extra.items()})
    accs = {(r, k): new_acc(sizes[k] if k != 'calibration' else len(cal_labels[r])) for r in methods for k in list(sizes) + ['calibration']}
    n, nb, d0, br = ctx['n_rows'], ctx['n_blk'], ctx['date0'], ctx['batch_rows']
    y_all, m_all, km_seg = ctx['y'], ctx['mask'], ctx['km']
    for lo in range(0, n, br):
        hi = min(n, lo + br)
        y = np.asarray(y_all[lo:hi], np.float64)
        if not (np.isfinite(y).all() and (y >= 0).all() and (y == np.floor(y)).all()):
            raise EvaluationError('labels must be finite non-negative integers')
        mask = np.asarray(m_all[lo:hi])
        d, s, b = cell_index(lo, hi, ctx['n_seg'], nb)
        d = d + d0
        pc = period_codes_for(periods, lo, hi)
        ps = {r: ctx['pred'][r](lo, hi) for r in methods}
        km = km_seg[s]
        col = assign_colour_codes(ps['selected'], km * 1000.0, ctx['contract'], unit).astype(np.int64)
        codes = {k: dim_codes(v, d, s, b, mask, y, pc) for k, v in dims.items()}
        codes.update(colour=col, floor=floor_codes(ps['baseline'], ps['selected'], floor), period_colour=pc * (GREY_CODE + 1) + col)
        for r, p in ps.items():
            dv = poisson_deviance_terms(y, p, floor)
            for k, c in codes.items():
                acc_add(accs[(r, k)], y, p, km, floor, c, dv)
            acc_add(accs[(r, 'calibration')], y, p, km, floor, calibration_codes(p, cal[r]), dv)
    rows = []
    for (r, k), a in accs.items():
        labels = dims[k][2] if k in dims else extra[k] if k in extra else cal_labels[r]
        rows += rows_from_acc(a, labels, unit=unit, scope=scope, role=r, method=methods[r], dimension=k)
    return pd.DataFrame(rows), accs


def province_reconciliation(S, met, unit, scope, n_rows):
    """Expected cells per province group from the INPUT segment province codes (axis order) x cells per segment, against the streamed
    province rows of every role: an 'unknown' group exists exactly when some input code is missing (zero expected unknown is reported as
    such, never forced), every known province keeps its full population and province records sum to the pooled records. It shares the
    label mapping (province_labels) with the dimension builder, so it checks the accumulation, not the mapping; it is not independent."""
    lab = np.asarray(province_labels(S)).astype(str)
    ns = lab.size
    if ns == 0 or n_rows % ns:
        raise EvaluationError('scope rows are not a whole number of cells per segment')
    per_seg = n_rows // ns
    keys, cnt = np.unique(lab, return_counts=True)
    expected = {str(k): int(c) * per_seg for k, c in zip(keys, cnt)}
    sub = met[(met['unit'] == unit) & (met['scope'] == scope)]
    prov, pooled = sub[sub['dimension'] == 'province'], sub[sub['dimension'] == 'pooled']
    probs = [] if len(prov) else ['no province rows']
    for role, d in prov.groupby('role', sort=True):
        got = {str(g): int(v) for g, v in zip(d['group'], d['n'])}
        if got != expected:
            diff = sorted(k for k in set(got) | set(expected) if got.get(k) != expected.get(k))
            probs.append(f'{role}: cells per province differ from the input codes for {diff[:10]}')
        tot = pooled.loc[pooled['role'] == role, 'total_actual']
        if len(tot) != 1 or not math.isclose(float(d['total_actual'].sum()), float(tot.iloc[0]), rel_tol=1e-12, abs_tol=1e-9):
            probs.append(f'{role}: province records do not sum to the pooled records')
    return {'unit': unit, 'scope': scope, 'segments': int(ns), 'cells_per_segment': int(per_seg),
            'known_province_groups': sum(k != 'unknown' for k in expected), 'expected_unknown_segments': int((lab == 'unknown').sum()),
            'expected_unknown_cells': expected.get('unknown', 0), 'unknown_group_present': bool((prov['group'].astype(str) == 'unknown').any()),
            'roles': sorted(prov['role'].unique().tolist()), 'problems': probs}


In [ ]:
# FUNCTIONS (3/4): the relative colour contract (reusable by a later notebook 22; pure functions, no IO), calibration bins and
# equal-road-km top-risk capture. Density = expected selected+linked recorded count / segment km for the SAME horizon; never a probability.
COLOUR_LABELS = ('base', 'yellow', 'orange', 'red')
GREY = 'grey'
GREY_CODE = len(COLOUR_LABELS)
CONTRACT_KIND = 'roadaware_relative_density_colour_contract'


def segment_density(pred, length_m):
    """(density records/km/period, valid mask). Invalid = non-finite or negative prediction, non-finite or non-positive length."""
    p, L = np.broadcast_arrays(np.asarray(pred, np.float64), np.asarray(length_m, np.float64))
    ok = np.isfinite(p) & (p >= 0) & np.isfinite(L) & (L > 0)
    dens = np.full(p.shape, np.nan)
    np.divide(p, L / 1000.0, out=dens, where=ok)
    return dens, ok


def density_digest(dens):
    return sha256_bytes(np.ascontiguousarray(np.asarray(dens, '<f8')).tobytes())


def build_colour_contract(ref_density, unit, ccfg, reference):
    """Cutoffs = quantiles of the unweighted full reference cells. Duplicate cutoffs are kept raw and collapse through the band rule."""
    d = np.asarray(ref_density, np.float64)
    if d.ndim != 1 or d.size == 0 or not (np.isfinite(d).all() and (d >= 0).all()):
        raise EvaluationError('reference densities must be a non-empty finite non-negative vector')
    qs = [float(q) for q in ccfg['quantiles']]
    if len(qs) != len(COLOUR_LABELS) - 1 or qs != sorted(qs):
        raise EvaluationError('need three increasing quantiles')
    raw = [float(v) for v in np.quantile(d, qs, method='linear')]
    codes = np.searchsorted(np.asarray(raw), d, side='left')
    counts = np.bincount(codes, minlength=len(COLOUR_LABELS))
    return {'kind': CONTRACT_KIND, 'version': ccfg['version'], 'unit': unit, 'density_definition': ccfg['density_definition'],
            'density_unit': 'records per km per ' + ('local day' if unit == 'daily' else '6 h block'), 'reference': {**reference, 'n_cells': int(d.size),
            'density_float64_le_sha256': density_digest(d), 'min': float(d.min()), 'max': float(d.max())}, 'quantiles': qs,
            'quantile_method': ccfg['quantile_method'], 'raw_cutoffs': raw, 'unique_cutoffs': sorted(set(raw)), 'assignment': ccfg['assignment'],
            'labels': list(COLOUR_LABELS), 'labels_in_use': [COLOUR_LABELS[i] for i in range(len(COLOUR_LABELS)) if counts[i] > 0],
            'reference_cell_counts': {COLOUR_LABELS[i]: int(counts[i]) for i in range(len(COLOUR_LABELS))},
            'unavailable_label': GREY, 'accessible_text': ccfg['accessible_text'], 'hex': ccfg['hex'], 'limitations': ccfg['limitations'],
            'fixed_across_routes_and_viewports': True, 'convertible_between_horizons': False}


def validate_colour_contract(c, unit):
    if not isinstance(c, dict) or c.get('kind') != CONTRACT_KIND:
        raise EvaluationError('not a colour contract')
    if c.get('unit') != unit:
        raise EvaluationError(f'colour contract is for {c.get("unit")!r}, not {unit!r}; thresholds are never converted between horizons')
    raw = c.get('raw_cutoffs')
    if not (isinstance(raw, list) and len(raw) == len(COLOUR_LABELS) - 1 and all(isinstance(v, float) and math.isfinite(v) and v >= 0 for v in raw)
            and raw == sorted(raw)):
        raise EvaluationError('raw_cutoffs must be three finite non-negative non-decreasing floats')
    if list(c.get('labels', [])) != list(COLOUR_LABELS):
        raise EvaluationError('unexpected labels')
    return np.asarray(raw, np.float64)


def assign_colour_codes(pred, length_m, contract, unit, supported=None):
    """int8 codes 0..3 (base..red) or 4 (grey). Band = number of raw cutoffs strictly below the density, so ties at a cutoff stay in the lower
    band and duplicate cutoffs skip labels. Grey for unsupported segments, invalid/missing predictions or lengths. Never 'safe'/green."""
    raw = validate_colour_contract(contract, unit)
    dens, ok = segment_density(pred, length_m)
    if supported is not None:
        ok = ok & np.broadcast_to(np.asarray(supported, bool), ok.shape)
    codes = np.full(dens.shape, GREY_CODE, np.int8)
    codes[ok] = np.searchsorted(raw, dens[ok], side='left').astype(np.int8)
    return codes


def colour_labels(codes):
    lab = np.array(list(COLOUR_LABELS) + [GREY], dtype=object)
    return lab[np.asarray(codes, np.int64)]


def colour_legend(contract):
    names = list(COLOUR_LABELS) + [GREY]
    return [{'label': n, 'hex': contract['hex'][n], 'text': contract['accessible_text'][n], 'in_use': n in contract['labels_in_use'] or n == GREY}
            for n in names]


def calibration_edges(pred, qedges):
    """Unique quantile edges of the predicted counts (unweighted cells); bins are [e_i, e_i+1) with the last bin closed."""
    e = np.unique(np.quantile(np.asarray(pred, np.float64), qedges, method='linear'))
    return e if e.size >= 2 else np.array([e[0], e[0]])


def calibration_codes(pred, edges):
    if edges.size < 2 or edges[0] == edges[-1]:
        return np.zeros(np.shape(pred), np.int64)
    return np.clip(np.searchsorted(edges, pred, side='right') - 1, 0, edges.size - 2).astype(np.int64)


def capture_by_period(dens, y, pred, km, fractions):
    """Equal road-km budget per period (row): rank segments by descending density (stable: ties keep ascending segment index), take whole
    segments while cumulative km <= f * total km, then the boundary segment's length fraction; its records are allocated proportionally to
    that fraction (disclosed assumption). Returns {f: (captured_actual, captured_pred)} per row, plus row totals."""
    dens, y, pred, km = (np.asarray(a, np.float64) for a in (dens, y, pred, km))
    if dens.shape != y.shape or dens.shape != pred.shape or dens.shape[1] != km.size or not (np.isfinite(dens).all() and (km > 0).all()):
        raise EvaluationError('capture needs finite densities and positive lengths with matching shapes')
    T, S = dens.shape
    order = np.argsort(-dens, axis=1, kind='stable')
    Ls = km[order]
    cum = np.concatenate([np.zeros((T, 1)), np.cumsum(Ls, axis=1)], axis=1)
    out = {}
    for arr_name, arr in (('actual', y), ('pred', pred)):
        srt = np.take_along_axis(arr, order, axis=1)
        cs = np.concatenate([np.zeros((T, 1)), np.cumsum(srt, axis=1)], axis=1)
        for f in fractions:
            budget = float(f) * km.sum()
            k = (cum[:, 1:] <= budget).sum(axis=1)
            kk = np.minimum(k, S - 1)
            rows = np.arange(T)
            frac = np.where(k < S, (budget - cum[rows, k]) / Ls[rows, kk], 0.0)
            out.setdefault(float(f), {})[arr_name] = cs[rows, k] + frac * srt[rows, kk] * (k < S)
    return out, y.sum(axis=1), pred.sum(axis=1)


def to_period_matrix(a, n_seg, n_blk):
    """Axis order date, segment, block -> rows = (date, block) periods, columns = segments."""
    a = np.asarray(a)
    return a.reshape(-1, n_seg, n_blk).transpose(0, 2, 1).reshape(-1, n_seg)


In [ ]:
# FUNCTIONS (4/4): charts from the saved aggregate tables only (PNG + SVG, matplotlib Agg, constrained layout so titles/notes never overlap
# panel titles) and the Thai results / Model Gate narrative. Every plotted value is a column of a CSV written in the same run; percentages
# are signed so that positive = selected (candidate) deviance lower.
import textwrap

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

UNIT_TITLE = {'daily': 'daily (per local day)', '6h': '6 h (per 6 h block)'}
KIND_COLOURS = {'routed_context': '#c62828', 'core': '#2c7fb8', 'context': '#b0b0b0', 'baseline': '#2e7d32'}
GATE_HEADINGS = ('Baseline และ Candidate Models', 'Evaluation Metrics', 'Experiment Results', 'Error Analysis', 'เหตุผลเลือกโมเดล')


def save_figure(fig, fig_dir, stem):
    names = []
    for ext in ('png', 'svg'):
        p = Path(fig_dir) / (stem + '.' + ext)
        if os.path.lexists(p):
            raise EvaluationError('refusing to overwrite ' + str(p))
        fig.savefig(p, dpi=150 if ext == 'png' else None, bbox_inches='tight')
        names.append('figures/' + p.name)
    plt.close(fig)
    return names


def _wrap(s, width):
    return '\n'.join(textwrap.fill(part, width) for part in str(s).split('\n'))


def _units_axes(units, w=6.4, h=4.2):
    fig, axes = plt.subplots(1, len(units), figsize=(w * len(units), h), squeeze=False, layout='constrained')
    return fig, axes[0]


def _titles(fig, title, note=None):
    """Suptitle is reserved by the constrained layout (never over panel titles); the note is a footer below the figure area."""
    width = int(fig.get_size_inches()[0] * 11)
    fig.suptitle(_wrap(title, width), fontsize=10)
    if note:
        fig.text(0.5, -0.01, _wrap(note, int(width * 1.15)), ha='center', va='top', fontsize=7.5, color='#333333')


def _val_labels(vals, pct):
    return ['' if not np.isfinite(v) else (f'{v:+.2f}%' if pct else f'{v:.3g}') for v in vals]


def fig_bars(df, units, x, y, title, ylabel, hue=None, pct=False, hline=None, note=None, horizontal=False, colour_by=None):
    n_cat = max([df.loc[df['unit'] == u, x].nunique() for u in units] + [1])
    fig, axes = _units_axes(units, h=max(3.6, 0.34 * n_cat + 1.8) if horizontal else 4.6)
    for ax, u in zip(axes, units):
        d = df[df['unit'] == u]
        hues = [None] if hue is None else list(dict.fromkeys(d[hue]))
        xs = list(dict.fromkeys(d[x]))
        w = 0.8 / len(hues)
        pos = np.arange(len(xs))
        for i, h in enumerate(hues):
            dd = d if h is None else d[d[hue] == h]
            vals = np.array([dd.loc[dd[x] == v, y].astype(float).mean() if (dd[x] == v).any() else np.nan for v in xs]) * (100 if pct else 1)
            colors = [KIND_COLOURS.get(d.loc[d[x] == v, colour_by].iloc[0], '#777777') for v in xs] if colour_by is not None and h is None else None
            draw = ax.barh if horizontal else ax.bar
            bars = draw(pos + i * w - 0.4 + w / 2, np.nan_to_num(vals), w, label=None if h is None else str(h), color=colors)
            if hue is None:
                ax.bar_label(bars, labels=_val_labels(vals, pct), fontsize=7, padding=2)
        if horizontal:
            ax.set_yticks(pos)
            ax.set_yticklabels([str(v) for v in xs], fontsize=8)
            ax.invert_yaxis()
            if hline is not None:
                ax.axvline(hline * (100 if pct else 1), color='k', ls='--', lw=0.8)
            ax.axvline(0, color='grey', lw=0.5)
            ax.set_xlabel(ylabel)
            ax.margins(x=0.18)
        else:
            ax.set_xticks(pos)
            ax.set_xticklabels([str(v) for v in xs], rotation=30, ha='right', fontsize=8)
            if hline is not None:
                ax.axhline(hline * (100 if pct else 1), color='k', ls='--', lw=0.8)
            ax.axhline(0, color='grey', lw=0.5)
            ax.set_ylabel(ylabel)
            ax.margins(y=0.15)
        ax.set_title(UNIT_TITLE.get(u, u), fontsize=10)
        if hue is not None:
            ax.legend(fontsize=7)
        elif colour_by is not None:
            kinds = list(dict.fromkeys(d[colour_by]))
            ax.legend(handles=[Patch(color=KIND_COLOURS.get(k, '#777777'), label=str(k)) for k in kinds], fontsize=7, loc='lower right')
    _titles(fig, title, note)
    return fig


def group_label(dim, group, names=None):
    g = str(group)
    if dim == 'province' and names and g in names:
        return g + ' ' + names[g]['name_en']
    return g


def group_gain_table(met, scope, dims):
    """Selected vs baseline per (unit, dimension, group) from metrics_by_group rows, in the evaluation's group order: cells, records, both mean
    deviances and deviance sums, relative reduction (1 - selected / baseline; None when the baseline deviance is 0). Empty groups
    (0 cells, e.g. a calendar label absent from the scope) are omitted."""
    m = met[(met['scope'] == scope) & met['dimension'].isin(list(dims))]
    key = ['unit', 'dimension', 'group']
    s = m[(m['role'] == 'selected') & (m['n'] > 0)].set_index(key)
    b = m[m['role'] == 'baseline'].set_index(key).reindex(s.index)
    if b['n'].isna().any() or not (b['n'].to_numpy(float) == s['n'].to_numpy(float)).all():
        raise EvaluationError('selected and baseline group rows differ')
    out = pd.DataFrame({'cells': s['n'].astype(int), 'records': s['total_actual'].astype(float),
                        'baseline_mean_deviance': b['mean_poisson_deviance'], 'selected_mean_deviance': s['mean_poisson_deviance'],
                        'baseline_deviance_sum': b['deviance_sum'], 'selected_deviance_sum': s['deviance_sum']}).reset_index()
    out['relative_deviance_reduction'] = [relative_reduction(x, z) for x, z in zip(out['baseline_mean_deviance'], out['selected_mean_deviance'])]
    out['relative_deviance_reduction'] = out['relative_deviance_reduction'].astype(float)
    return out


def fig_group_panels(gg, units, dims, title, note=None, names=None):
    """One horizontal-bar panel per dimension x unit; each label carries its support (cells, records)."""
    counts = [max([len(gg[(gg['unit'] == u) & (gg['dimension'] == k)]) for u in units] + [1]) for k in dims]
    fig, axes = plt.subplots(len(dims), len(units), figsize=(7.2 * len(units), 0.27 * sum(counts) + 1.0 * len(dims) + 1.2), squeeze=False,
                             layout='constrained', gridspec_kw={'height_ratios': [c + 2 for c in counts]})
    for i, k in enumerate(dims):
        for j, u in enumerate(units):
            ax = axes[i][j]
            d = gg[(gg['unit'] == u) & (gg['dimension'] == k)]
            vals = d['relative_deviance_reduction'].astype(float).to_numpy() * 100
            pos = np.arange(len(d))
            bars = ax.barh(pos, np.nan_to_num(vals), 0.7, color=['#2c7fb8' if v >= 0 else '#c62828' for v in np.nan_to_num(vals)])
            ax.bar_label(bars, labels=_val_labels(vals, True), fontsize=6.5, padding=2)
            ax.set_yticks(pos)
            ax.set_yticklabels([f'{group_label(k, g, names)}  (n={int(c):,}; rec={int(r):,})' for g, c, r in zip(d['group'], d['cells'], d['records'])], fontsize=7)
            ax.invert_yaxis()
            ax.axvline(0, color='grey', lw=0.6)
            ax.margins(x=0.22)
            ax.set_title(f'{UNIT_TITLE.get(u, u)}: {k}', fontsize=9)
            if i == len(dims) - 1:
                ax.set_xlabel('% deviance reduction, selected vs baseline (positive = selected lower)', fontsize=8)
    _titles(fig, title, note)
    return fig


def fig_calibration(cal, units):
    fig, axes = _units_axes(units, h=4.6)
    for ax, u in zip(axes, units):
        for m, d in cal[cal['unit'] == u].groupby('role', sort=False):
            ax.loglog(d['predicted_records_per_km_period'], d['observed_records_per_km_period'], 'o-', label=m)
        lim = cal.loc[cal['unit'] == u, ['predicted_records_per_km_period', 'observed_records_per_km_period']].replace(0, np.nan).stack()
        if len(lim):
            ax.plot([lim.min(), lim.max()], [lim.min(), lim.max()], 'k--', lw=0.8, label='perfect calibration')
        ax.set_xlabel('predicted records per segment-km-period (bin total / exposure)', fontsize=8)
        ax.set_ylabel('observed records per segment-km-period', fontsize=8)
        ax.set_title(UNIT_TITLE[u], fontsize=10)
        ax.legend(fontsize=7)
    _titles(fig, 'Calibration by predicted-count bin, 2022-23 OOF (exposure-weighted totals)',
            'bins = predicted-count quantiles per role; supports per bin in figure_calibration.csv (n, total_actual)')
    return fig


def fig_capture(cap, units):
    fig, axes = _units_axes(units, h=4.6)
    for ax, u in zip(axes, units):
        d = cap[(cap['unit'] == u) & (cap['scope'] == 'oof_2022_2023') & (cap['time_block'] == 'all')]
        for m, dd in d.groupby('role', sort=False):
            ax.plot(dd['road_km_fraction'] * 100, dd['capture_share'] * 100, 'o-', label=m + ' ranking')
        f = sorted(d['road_km_fraction'].unique())
        ax.plot([v * 100 for v in f], [v * 100 for v in f], 'k--', lw=0.8, label='uniform road-km expectation')
        ax.set_xlabel('top road-km budget per period (% of network km)', fontsize=8)
        ax.set_ylabel('% of recorded+linked records captured (pooled)', fontsize=8)
        ax.set_title(UNIT_TITLE[u], fontsize=10)
        ax.legend(fontsize=7)
    _titles(fig, 'Equal-road-km capture per day / 6 h block, 2022-23 OOF (descriptive, not a safety benefit)',
            'boundary segment records allocated by length fraction (assumption, not observed partial-segment events)')
    return fig


def fmt(v, nd=4, pct=False):
    if v is None or (isinstance(v, (float, np.floating)) and not math.isfinite(v)):
        return 'n/a'
    return f'{v * 100:.{nd}f}%' if pct else f'{v:.{nd}g}'


def md_table(df, cols):
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    for _, r in df[cols].iterrows():
        lines.append('| ' + ' | '.join(fmt(v) if isinstance(v, float) else str(v) for v in r) + ' |')
    return '\n'.join(lines)


def _ok(v):
    return v is not None and not (isinstance(v, (float, np.floating)) and not math.isfinite(v))


def _p(v, nd=3, signed=True):
    return 'n/a' if not _ok(v) else (f'{v * 100:+.{nd}f}%' if signed else f'{v * 100:.{nd}f}%')


def _n(v, nd=6):
    return 'n/a' if not _ok(v) else f'{float(v):.{nd}f}'


def _c(v):
    return 'n/a' if not _ok(v) else f'{int(round(float(v))):,}'


def _tab(header, rows):
    return '\n'.join(['| ' + ' | '.join(header) + ' |', '|' + '---|' * len(header)] + ['| ' + ' | '.join(str(x) for x in r) + ' |' for r in rows])


def figure_links(figs, *stems):
    return ', '.join(f'[{Path(n).name}]({n})' for n in figs if Path(n).stem in stems) or '(no figure)'


def results_th(sel_rows, floor_sum, contracts, cap, diag_rows, transfer_rows, boot):
    out = ['# ผลประเมิน notebook 21: ความคลาดเคลื่อนและเกณฑ์สีความหนาแน่นสัมพัทธ์ (V3)', '',
           'ข้อมูลทั้งหมดคำนวณใหม่จาก array OOF 2022-23 ของ notebook 20 ที่ผ่านการยอมรับแล้ว (ไม่ฝึกใหม่ ไม่เลือกโมเดลใหม่ ไม่ใช้ข้อมูล 2026)',
           'label = จำนวนบันทึก MOT ที่ถูกเลือกและเชื่อมกับช่วงถนน DOH ได้ ไม่ใช่จำนวนอุบัติเหตุจริงทั้งหมด และไม่ใช่ความน่าจะเป็นต่อคันรถ', '',
           '## 1. เหตุผลการเลือก (gate ที่ล็อกไว้)', '']
    for r in sel_rows:
        out.append(f"- {r['unit']}: เลือก `{r['selected_method']}` เทียบ baseline `{r['baseline']}`; deviance ลดลง {fmt(r['relative_deviance_reduction'], 4, True)}, "
                   f"MAE ratio {fmt(r['mae_ratio'], 7)}, ชนะ {r['n_fold_wins']}/4 fold, ทุก gate ผ่าน = {r['all_pass']}, คง baseline = {r['baseline_retained']}")
    out += ['', '## 2. ข้อจำกัด: กลุ่มที่ baseline ทำนายต่ำกว่า floor ของ metric', '']
    for s in floor_sum:
        out.append(f"- {s['unit']} ({s['scope']}): ส่วนแบ่งของ gain จากเซลล์ที่ baseline < floor = {fmt(s['baseline_floor_gain_share'], 4, True)}; "
                   f"gain ในเซลล์ที่เหลือ = {fmt(s['complement_relative_deviance_reduction'], 4, True)} -- ไม่ใช่หลักฐานว่าโมเดลดีกว่าในทุกกรณี")
    out += ['', 'audit นี้ลงทะเบียนหลังเห็นผล daily เบื้องต้น จึงเป็น sensitivity เท่านั้น ไม่เปลี่ยน baseline/metric/การเลือก', '',
            '## 3. เกณฑ์สี (คงที่ทุกเส้นทาง/ทุกมุมมอง, แยก daily กับ 6h)', '']
    for u, c in contracts.items():
        out.append(f"- {u}: cutoffs P50/P80/P95 = {', '.join(repr(v) for v in c['raw_cutoffs'])} ({c['density_unit']}); สีที่ใช้จริง: {', '.join(c['labels_in_use'])}")
    out += ['', 'สี base = ความหนาแน่นที่คาดการณ์ "ต่ำกว่าเชิงสัมพัทธ์" เท่านั้น ไม่ได้แปลว่าปลอดภัย; สีเทา = ไม่มีข้อมูล/ไม่รองรับ', '',
            '## 4. Capture ที่งบ road-km เท่ากัน (OOF 2022-23, รวมทุกช่วงเวลา)', '',
            md_table(cap[(cap['scope'] == 'oof_2022_2023') & (cap['time_block'] == 'all')], ['unit', 'role', 'road_km_fraction', 'capture_share', 'lift_vs_uniform']), '',
            '## 5. ความเสถียร 2024/25 (เคยเห็นแล้ว, ไม่ใช่ holdout) และ spatial transfer', '',
            md_table(diag_rows, ['unit', 'group', 'role', 'mean_poisson_deviance', 'mae', 'pred_over_actual']), '',
            md_table(transfer_rows, ['unit', 'method', 'mean_poisson_deviance', 'relative_to_best_baseline']), '',
            'Transfer ประเมินเฉพาะ core config บนช่วงถนนที่ไม่ได้ใช้ฝึก (2023) แยกจากการเลือกเชิงเวลา; ถ้าผลแย่ต้องรายงานตามจริง', '',
            '## ความไม่แน่นอน (exploratory, หลังการเลือก)', '']
    for u, b in boot.items():
        out.append(f"- {u}: paired day-block bootstrap ของ notebook 20: {fmt(b.get('ci_low'), 4, True)} ถึง {fmt(b.get('ci_high'), 4, True)} ({b.get('ci_status')})")
    out += ['', '2026: รอการตรวจข้อมูลและการประเมินแยกต่างหาก (ยังไม่มีผล ไม่มีการเติมตัวเลขสมมติ)', '']
    return '\n'.join(out)


FAMILY_TH = {
    'poisson': 'Poisson regression (GLM, log link): log μ = β0 + Σ βj·xj; ผลของแต่ละ feature เป็นแบบทวีคูณต่อค่าคาดหมาย; alpha = L2 penalty',
    'random_forest': 'Random Forest (criterion=poisson): μ = ค่าเฉลี่ยของหลายต้นไม้ที่แบ่งกลุ่มแบบไม่เชิงเส้น; แต่ละต้นใช้ bootstrap ของแถวและสุ่ม feature',
    'hist_gradient_boosting': 'Histogram Gradient Boosting (loss=poisson): log μ = ผลรวมของต้นไม้ที่เพิ่มทีละต้นเพื่อลด Poisson loss ที่เหลือ; learning_rate คุมขนาดก้าว',
}


def _baseline_definition(name, defs):
    if name in defs:
        return defs[name]
    if name.startswith('segment_mean_k') and 'segment_mean_k' in defs:
        return defs['segment_mean_k'] + ' with k = ' + name[len('segment_mean_k'):]
    return 'see notebook-20 config'


def presentation_th(P):
    """Thai Model Gate content in exactly the five GATE_HEADINGS sections (slide order + speaker points inside them), built only from tables
    of this run and the accepted notebook-20 config/lock. P keys: label_meaning, sel_rows, config20, feature_columns, leaderboard (strongest
    baseline + candidate policies), leaderboard_all, met, floor_rows, floor_sum, cap, colour_rows, transfer, boot, contracts, figs,
    province_names, network {segments, network_km}, diag_context_cells {unit: int}, capture_cfg."""
    sel, met, c20, lb, lba = P['sel_rows'], P['met'], P['config20'], P['leaderboard'], P['leaderboard_all']
    units = [r['unit'] for r in sel]
    names, figs = P.get('province_names') or {}, P['figs']
    pooled = {(u, r): met[(met.unit == u) & (met.scope == 'oof') & (met.role == r) & (met.dimension == 'pooled')].iloc[0] for u in units for r in ('selected', 'baseline')}
    nseg = int(P['network']['segments'])
    out = ['# RoadAware Model Gate: เนื้อหานำเสนอ 5 หัวข้อ (เนื้อหา/asset ไม่ใช่ไฟล์ PowerPoint)', '',
           'สร้างอัตโนมัติจากผลจริงของรอบนี้ (notebook 21) บน OOF 2022-23 (ใช้เลือก) และ 2024/25 (previously seen diagnostics ไม่ใช่ holdout) ของ notebook 20 '
           'ที่ตรวจรับแล้ว; ไม่ฝึกใหม่ ไม่เลือกใหม่. **2026: รอการตรวจข้อมูลและการประเมินแยกต่างหาก ไม่มีคะแนน 2026 ในเอกสารนี้**', '',
           'ลำดับสไลด์: (1) target และพื้นที่ -> (2) baselines -> (3) candidates/features -> (4) metrics และ gate -> (5) ผล OOF -> '
           '(6) folds / 2024-25 / transfer -> (7) error: totals, calibration, floor -> (8) กลุ่มที่ baseline ดีกว่า -> (9) capture และสี -> (10) เหตุผลเลือกและข้อจำกัด', '']

    # 1. Baselines and candidates
    out += ['## ' + GATE_HEADINGS[0], '', '### สไลด์ 1: target และขอบเขตพื้นที่ที่รองรับ', '', str(P['label_meaning']), '',
            f"- Universe: {nseg:,} ช่วงถนน DOH ({P['network']['network_km']:,.1f} km จาก geom_length_m); OOF 2022-23: "
            + '; '.join(f"{u} {_c(pooled[(u, 'selected')]['n'])} cells / {_c(pooled[(u, 'selected')]['total_actual'])} records" for u in units), '']
    u0 = units[0]
    pr = met[(met.unit == u0) & (met.scope == 'oof') & (met.role == 'selected') & (met.dimension == 'province')].copy()
    per_seg = int(pooled[(u0, 'selected')]['n']) // nseg if nseg else 0
    pr['segments'] = pr['n'] // per_seg if per_seg else 0
    pr = pr.sort_values(['segments', 'group'], ascending=[False, True])
    rows = [(g, names.get(str(g), {}).get('name_th', '-'), names.get(str(g), {}).get('name_en', '-'), _c(s), _c(a)) for g, s, a in zip(pr['group'], pr['segments'], pr['total_actual'])]
    out += [f'ตัวอย่างพื้นที่ที่รองรับ (province ของจุดกึ่งกลาง segment ตาม geometry ปัจจุบัน ไม่ใช่การจัดจังหวัดจากเหตุการณ์; ชื่อจาก province_points.csv ตาม province_code):', '',
            _tab(['province_code', 'จังหวัด', 'Province', 'segments', f'OOF records ({u0})'], rows), '',
            '- segment นอก universe / ไม่รองรับ / prediction หรือความยาวไม่ถูกต้อง = สีเทา unavailable; ไม่สร้าง prediction 0 ทดแทน', '',
            '> ประเด็นพูด: เราทำนายค่าคาดหมายจำนวนรายการที่ฐานข้อมูลคัดเลือกและเชื่อมถนนได้ ไม่ใช่โอกาสชนรายบุคคล; ศูนย์ = ไม่มีบันทึก ไม่ใช่ไม่มีอุบัติเหตุ', '']
    defs, inel = c20['baselines']['definitions'], set(c20['baselines'].get('ineligible', []))
    brow = []
    for b in c20['baselines']['order']:
        cells = []
        for u in units:
            r = lba[(lba.unit == u) & (lba.method == b)]
            cells += [_n(r.mean_poisson_deviance.iloc[0]) if len(r) else 'n/a', _n(r.mae.iloc[0]) if len(r) else 'n/a']
        brow.append([f'`{b}`' + (' (ineligible)' if b in inel else ''), _baseline_definition(b, defs)] + cells)
    out += ['### สไลด์ 2: Baselines (ตัวหาร = ปฏิทิน training เต็ม ไม่ใช่ training sample)', '', defs.get('notation', ''), '',
            _tab(['baseline', 'นิยาม'] + [f'{u} {k}' for u in units for k in ('OOF deviance', 'OOF MAE')], brow), '',
            '- strongest nonzero baseline ตาม pooled OOF deviance: ' + '; '.join(f"{r['unit']} `{r['baseline']}`" for r in sel),
            '- parent: ' + defs.get('parent', '') + '; ' + defs.get('floor', ''), '',
            '> ประเด็นพูด: ข้อมูล sparse มาก baseline ที่ shrink ไปหา parent ของสายทางจึงแข็งแรง; zero baseline เป็นตัวเปรียบเทียบเชิงวินิจฉัยเท่านั้น', '']
    cand = c20['candidates']
    out += ['### สไลด์ 3: Candidate models, policies และ features', '',
            _tab(['config', 'family', 'locked params'], [(f"`{c['config_id']}`", c['family'], '`' + json.dumps(c['params'], sort_keys=True) + '`') for c in cand['configs']]), '']
    out += [f"- {f}: {FAMILY_TH.get(f, f)}; preprocessing: {cand['preprocessing'].get(f, '')}" for f in cand['families']]
    out += [f"- policy `{k}`: {v}" + ('' if k in cand['eligible_policies'] else ' (ไม่ eligible)') for k, v in cand['policies'].items()]
    fc = P['feature_columns']
    for u in units:
        core, ctx = fc[u][cand['core_variant']], fc[u][cand['context_variant']]
        out += [f"- {u} core features ({len(core)}): {', '.join(core)}",
                f"- {u} context เพิ่ม ({len(ctx) - len(core)}): {', '.join(c for c in ctx if c not in core)}; routed trigger (ต้องครบทุกค่า): {', '.join(cand['routed_trigger_columns'][u])}"]
    out += [f"- training sample: {c20['sampling']['use']}; weights positive={c20['sampling']['weights']['positive']}, zero={c20['sampling']['weights']['zero']}",
            '- ไม่มี segment_id / target / event coordinates / province เป็น predictor (allowlist ของ notebook 19/20); AADT = annual survey proxy ไม่ใช่ live congestion', '',
            '> ประเด็นพูด: สามตระกูล Poisson/RF/HGB ใช้ config ที่ล็อกก่อนดูคะแนน; routed_context ใช้ context เมื่อ weather+AADT ครบ มิฉะนั้นใช้ core ของ config เดียวกัน', '']

    # 2. Metrics
    s20, fl = c20['selection'], c20['metrics']['deviance_floor']
    folds = ', '.join(f"{f['fold_id']} ({f['start']}..{f['end']})" for f in c20['time']['folds'])
    out += ['## ' + GATE_HEADINGS[1], '', '### สไลด์ 4: นิยาม metric และ gate ที่ล็อก', '',
            f'- Mean Poisson deviance = (1/N) Σ 2[y·ln(y/m) − (y − m)], m = max(μ, {fl!r}) เฉพาะใน metric (y·ln = 0 เมื่อ y = 0); ยิ่งต่ำยิ่งดี ไม่ใช่ accuracy',
            '- MAE = (1/N) Σ|μ − y|; RMSE = √((1/N) Σ(μ − y)²) หน่วย records ต่อ segment-period; bias = (1/N) Σ(μ − y) (ลบ = ทำนายต่ำ)',
            '- WAPE = Σ|μ − y| / Σy; กลุ่มที่ Σy = 0 → undefined (ไม่ใช่ 0 หรือ ∞)',
            '- pooled OOF = ผลรวม loss / จำนวน cells ทั้งหมด (row-weighted ไม่เฉลี่ย fold เท่ากัน); daily และ 6h คนละ horizon/population ห้ามเทียบคะแนนข้ามหน่วย',
            f"- folds: {folds}; label latency {c20['time']['label_latency_days']} วัน ({c20['time']['label_latency_note']})",
            f"- Gate: deviance ลด ≥ {_p(s20['min_relative_deviance_reduction'], 1, False)}, MAE ratio ≤ {s20['max_mae_ratio']}, ชนะ deviance ≥ {s20['min_fold_wins']}/{s20['n_folds']} folds; {s20['rule']}",
            '- density สี = expected records / segment km ต่อ horizon เดียวกัน; cutoffs = P50/P80/P95 (type 7) ของ OOF ทั้งหมดแบบ unweighted; calibration = ผลรวม pred / ผลรวม actual ต่อ exposure (segment-km-periods)',
            f"- capture: {P['capture_cfg']['tie_break']}; {P['capture_cfg']['boundary']}",
            '- 2024/25 ใช้หลังเลือกแล้วเป็น diagnostics เท่านั้น; 2026 จะไม่ใช้เปลี่ยน winner', '',
            '> ประเด็นพูด: เลือกด้วย temporal OOF เท่านั้น ไม่มี random split; ค่า gate ถูกล็อกก่อนเห็นคะแนน', '']

    # 3. Results
    out += ['## ' + GATE_HEADINGS[2], '', '### สไลด์ 5: ผล OOF 2022-23 (ใช้เลือก)', '']
    for u in units:
        d = lb[lb.unit == u].sort_values('mean_poisson_deviance')
        out += [f'**{UNIT_TITLE.get(u, u)}** (n = {_c(d.n.iloc[0]) if len(d) else "n/a"} cells ต่อ method)', '',
                _tab(['method', 'kind', 'eligible', 'deviance', 'MAE', 'RMSE', 'deviance reduction vs strongest baseline'],
                     [(f'`{r.method}`', r.kind, r.deployment_eligible, _n(r.mean_poisson_deviance), _n(r.mae), _n(r.rmse), _p(r.relative_deviance_reduction_vs_strongest_baseline))
                      for r in d.itertuples()]), '']
    out += [_tab(['unit', 'selected', 'baseline', 'deviance reduction', 'MAE ratio', 'fold wins', 'all gates pass', 'baseline retained'],
                 [(r['unit'], f"`{r['selected_method']}`", f"`{r['baseline']}`", _p(r['relative_deviance_reduction']), _n(r['mae_ratio'], 6),
                   f"{r['n_fold_wins']}/{s20['n_folds']}", r['all_pass'], r['baseline_retained']) for r in sel]), '']
    out += [f"- {u}: paired 7-day block bootstrap ({b.get('resamples')} resamples) ช่วง {_p(b.get('ci_low'))} ถึง {_p(b.get('ci_high'))} — {b.get('ci_status')}" for u, b in P['boot'].items()]
    out += ['', 'รูป: ' + figure_links(figs, 'method_comparison'), '',
            '> ประเด็นพูด: บอก gain พร้อม MAE trade-off; context (สีเทา) เป็น ablation ไม่ eligible; ไม่เรียกว่า accuracy', '']

    def period_rows(scope):
        g = group_gain_table(met, scope, ['period'])
        return [(r.unit, r.group, _c(r.cells), _c(r.records), _n(r.baseline_mean_deviance), _n(r.selected_mean_deviance), _p(r.relative_deviance_reduction)) for r in g.itertuples()]
    hdr = ['unit', 'period', 'cells', 'records', 'baseline deviance', 'selected deviance', 'reduction']
    out += ['### สไลด์ 6: ความเสถียร: folds, 2024/25 ที่เคยเห็นแล้ว และ spatial transfer', '', '**OOF folds (ใช้เลือก)**', '', _tab(hdr, period_rows('oof')), '',
            '**2024/25: previously seen diagnostics (ไม่ใช่ untouched test; ไม่ใช้เลือก)**', '', _tab(hdr, period_rows('diag')), '',
            '- context-available cells 2024/25: ' + '; '.join(f'{u} {v:,}' for u, v in P['diag_context_cells'].items())
            + ' (0 = selected routed policy ใช้ core fallback ตลอดช่วง)', '']
    tr = P['transfer']
    out += ['**Spatial transfer 2023 (held-out sections; แยกจากการเลือกเชิงเวลา)**', '',
            _tab(['unit', 'method', 'cells', 'records', 'deviance', 'vs best transfer baseline'],
                 [(r.unit, f'`{r.method}`', _c(r.n), _c(r.total_actual), _n(r.mean_poisson_deviance), _p(r.relative_to_best_baseline)) for r in tr.itertuples()]), '',
            'รูป: ' + figure_links(figs, 'fold_performance', 'seen_2024_2025', 'spatial_transfer'), '',
            '> ประเด็นพูด: 2024/25 และ transfer เป็นหลักฐานเสริม ไม่ใช่ holdout; transfer ประเมิน core เท่านั้นในพื้นที่เดียว. 2026 ยังรอการตรวจข้อมูล', '']

    # 4. Error analysis
    out += ['## ' + GATE_HEADINGS[3], '', '### สไลด์ 7: ยอดรวม, calibration และ metric floor', '',
            _tab(['unit', 'role', 'total predicted', 'total actual', 'pred / actual', 'bias per cell'],
                 [(u, r, _n(pooled[(u, r)]['total_pred'], 2), _c(pooled[(u, r)]['total_actual']), _n(pooled[(u, r)]['pred_over_actual'], 4), _n(pooled[(u, r)]['bias'], 7))
                  for u in units for r in ('selected', 'baseline')]), '']
    crow = []
    for u in units:
        for r in ('selected', 'baseline'):
            c = met[(met.unit == u) & (met.scope == 'oof') & (met.role == r) & (met.dimension == 'calibration') & (met.n > 0)]
            if len(c):
                t = c.iloc[-1]
                crow.append((u, r, t['group'], _c(t['n']), _c(t['total_actual']), _n(t['total_pred'] / t['n'], 5), _n(t['total_actual'] / t['n'], 5)))
    out += ['calibration bin สูงสุด (support จริงของ bin):', '', _tab(['unit', 'role', 'predicted-count bin', 'cells', 'records', 'mean predicted', 'mean actual'], crow), '']
    out += [_tab(['unit', 'floor group', 'cells', 'actual-positive cells', 'records', 'share of pooled gain'],
                 [(r.unit, r.floor_group, _c(r.cells), _c(r.actual_positive_cells), _c(r.actual_records), _p(r.share_of_total_gain, 2, False)) for r in P['floor_rows'].itertuples()]), '']
    out += [f"- {s['unit']}: gain จากเซลล์ที่ baseline < floor = {_p(s['baseline_floor_gain_share'], 2, False)}; reduction ในเซลล์ที่เหลือ = {_p(s['complement_relative_deviance_reduction'])} "
            '(sensitivity ที่ลงทะเบียนหลังเห็นผล daily เบื้องต้น ไม่ใช่ gate)' for s in P['floor_sum']]
    out += ['', 'รูป: ' + figure_links(figs, 'calibration', 'floor_audit'), '',
            '> ประเด็นพูด: ยอดรวมยังทำนายต่ำ; gain ส่วนใหญ่มาจากเซลล์ที่ baseline ทำนาย 0 — รายงานตรงไปตรงมา ไม่เปลี่ยน baseline/metric ย้อนหลัง', '']
    dims = ['actual_zero', 'context', 'time_block', 'season', 'weekday', 'month', 'festival', 'province']
    gg = group_gain_table(met, 'oof', dims + ['route_code'])
    neg = gg[(gg.dimension != 'route_code') & (gg.relative_deviance_reduction < 0)].sort_values(['unit', 'relative_deviance_reduction'])
    out += ['### สไลด์ 8: กลุ่มที่ baseline ดีกว่า (OOF 2022-23, descriptive หลังการเลือก ไม่ใช่ gate ใหม่)', '',
            _tab(['unit', 'dimension', 'group', 'cells', 'records', 'baseline deviance', 'selected deviance', 'reduction'],
                 [(r.unit, r.dimension, group_label(r.dimension, r.group, names), _c(r.cells), _c(r.records), _n(r.baseline_mean_deviance), _n(r.selected_mean_deviance),
                   _p(r.relative_deviance_reduction, 4)) for r in neg.itertuples()]) if len(neg) else 'ไม่มีกลุ่มที่ baseline ดีกว่าใน dimension เหล่านี้', '']
    for u in units:
        rc = gg[(gg.unit == u) & (gg.dimension == 'route_code')]
        out.append(f'- {u}: route_code ที่ baseline ดีกว่า {int((rc.relative_deviance_reduction < 0).sum())}/{len(rc)} สายทาง (support ต่อสายทางน้อย; ดู metrics_by_group.csv)')
    out += ['- province = จุดกึ่งกลาง segment ตาม geometry ปัจจุบัน ไม่ใช่ province ของเหตุการณ์; กลุ่ม support น้อยมีความไม่แน่นอนสูง', '',
            'รูป: ' + figure_links(figs, 'errors_by_group', 'errors_by_area_month'), '',
            '> ประเด็นพูด: โมเดลชนะโดยรวมแต่ไม่ชนะทุกกลุ่ม; แสดงจำนวน cells/records ทุกกลุ่ม ห้ามใช้กลุ่มเหล่านี้เปลี่ยน winner หรือ threshold', '']
    cap = P['cap']
    cap = cap[(cap.scope == 'oof_2022_2023') & (cap.time_block == 'all')]
    krow, better = [], {}
    for u in units:
        for f in sorted(cap.road_km_fraction.unique()):
            s = cap[(cap.unit == u) & (cap.role == 'selected') & (cap.road_km_fraction == f)].capture_share
            b = cap[(cap.unit == u) & (cap.role == 'baseline') & (cap.road_km_fraction == f)].capture_share
            if len(s) and len(b):
                w = 'selected' if s.iloc[0] > b.iloc[0] else 'baseline' if b.iloc[0] > s.iloc[0] else 'tie'
                better.setdefault(u, []).append((f, w))
                krow.append((u, _p(f, 0, False), _p(s.iloc[0], 2, False), _p(b.iloc[0], 2, False), _p(f, 0, False), w))
    out += ['### สไลด์ 9: capture ที่งบ road-km เท่ากัน และสีความหนาแน่นสัมพัทธ์', '',
            _tab(['unit', 'top road-km', 'selected capture', 'baseline capture', 'uniform', 'better'], krow), '',
            '- boundary segment ถูกนับ records ตามสัดส่วนความยาว (สมมติฐาน ไม่ใช่ตำแหน่งเหตุจริงในเศษ segment); ไม่ใช่ผลด้านความปลอดภัยเชิงสาเหตุ', '',
            _tab(['unit', 'P50 / P80 / P95 cutoffs', 'density unit', 'labels in use', 'reference cells per band'],
                 [(u, ' / '.join(f'{v:.6g}' for v in c['raw_cutoffs']), c['density_unit'], ', '.join(c['labels_in_use']),
                   ', '.join(f'{k} {v:,}' for k, v in c['reference_cell_counts'].items())) for u, c in P['contracts'].items()]), '']
    col = P['colour_rows']
    col = col[col.scope == 'oof_2022_2023']
    out += [_tab(['unit', 'colour', 'cells', 'cell share', 'exposure share', 'record share', 'observed records per km-period'],
                 [(r.unit, r.colour, _c(r.cells), _p(r.cell_share, 2, False), _p(r.exposure_share, 2, False), _p(r.record_share, 2, False), _n(r.observed_records_per_km_period, 6))
                  for r in col.itertuples()]), '',
            'รูป: ' + figure_links(figs, 'capture_curve', 'colour_exposure_capture'), '',
            '> ประเด็นพูด: base = ต่ำกว่าเชิงสัมพัทธ์ ไม่ใช่ปลอดภัย; cutoffs คงที่ทุกเส้นทาง/มุมมอง ไม่แปลงข้าม daily/6h; การจัดอันดับไม่ชนะทุกงบ road-km', '']

    # 5. Reason
    out += ['## ' + GATE_HEADINGS[4], '', '### สไลด์ 10: เหตุผลเลือกและข้อจำกัด', '']
    for r in sel:
        u = r['unit']
        d = lba[(lba.unit == u) & (lba.deployment_eligible.astype(str) == 'True')]
        me = d[d.method == r['selected_method']]
        lower = lambda k: [f'`{m}`' for m, v in zip(d.method, d[k]) if len(me) and v < me[k].iloc[0]]
        fs = [s for s in P['floor_sum'] if s['unit'] == u]
        nb = int(((neg.unit == u)).sum())
        decision = 'ผ่านทุก gate จึงเลือก candidate' if not r['baseline_retained'] else 'ไม่ผ่าน gate จึงคง baseline (ไม่อ้าง gain ของ candidate)'
        out += [f"- **{u}**: `{r['selected_method']}` เทียบ `{r['baseline']}` — {decision}: deviance {_p(r['relative_deviance_reduction'])}, MAE ratio {_n(r['mae_ratio'], 6)} "
                f"(MAE เปลี่ยน {_p(r['mae_ratio'] - 1.0)}), ชนะ {r['n_fold_wins']}/{s20['n_folds']} folds",
                f"  - trade-off: eligible methods ที่ MAE ต่ำกว่า: {', '.join(lower('mae')) or 'ไม่มี'}; RMSE ต่ำกว่า: {', '.join(lower('rmse')) or 'ไม่มี'}",
                f"  - gain จากเซลล์ baseline < floor: {_p(fs[0]['baseline_floor_gain_share'], 2, False) if fs else 'n/a'}; กลุ่ม (ไม่รวม route) ที่ baseline ดีกว่า: {nb}",
                f"  - capture: selected ดีกว่าที่ {', '.join(_p(f, 0, False) for f, w in better.get(u, []) if w == 'selected') or '-'} road-km; "
                f"baseline ดีกว่าที่ {', '.join(_p(f, 0, False) for f, w in better.get(u, []) if w == 'baseline') or '-'}"]
    out += ['', '- ไม่อ้างว่าดีที่สุดสำหรับทุก ranking task / ทุก metric / ทุกกลุ่ม; ไม่ใช่ untouched test และยังไม่มี production/safety certification',
            '- ขอบเขตที่ไม่รองรับตอบ unsupported/unavailable (สีเทา) ไม่สร้าง prediction 0 ทดแทน; แนะนำ benchmark-only serving ของ bundle notebook 20 จนกว่าจะมีความต้องการผู้ใช้แยก',
            '- 2026: รอการตรวจข้อมูล (coverage/lag/eligibility) และการประเมินแยก — ยังไม่มีผล ห้ามเติมตัวเลข และจะไม่ใช้เปลี่ยน winner', '',
            '> ประเด็นพูด: เลือกตาม gate ที่ล็อกโดยไม่ปรับเกณฑ์หลังเห็นคะแนน; ระบุจุดอ่อน (MAE, ยอดรวมต่ำ, floor share, กลุ่มที่แพ้) ให้ชัด', '']
    return '\n'.join(out)


In [ ]:
# SETUP: config, dependency pins, output guard BEFORE any write, protected hashes, check/stage/put helpers.
import contextlib
import resource

STARTED, T_START = utcnow(), time.time()
cfg = read_strict_json(CONFIG_PATH)
CFG_SHA = sha256_file(CONFIG_PATH)
VERSIONS = require_modules(cfg['dependency_pins'])
PATHS = output_paths(OUTPUT_ROOT, cfg)
_probs = output_problems(PATHS, BACKEND, cfg)
if _probs:
    raise EvaluationError('output guard refused (nothing written): ' + '; '.join(_probs))
PROTECTED_BEFORE = hash_paths(BACKEND, cfg['protected_paths'])
CHECKS, DURATIONS, REPORT_FILES, DATA_FILES = [], {}, {}, {}
UNITS = cfg['units']
FLOOR, BR = float(cfg['metrics']['deviance_floor']), int(cfg['metrics']['batch_rows'])
RT, AT, KEYS = cfg['metrics']['reconcile_rtol'], cfg['metrics']['reconcile_atol'], cfg['metrics']['keys']
PATHS['report_dir'].mkdir(parents=True, exist_ok=False)
PATHS['figure_dir'].mkdir(exist_ok=False)
PATHS['data_dir'].mkdir(parents=True, exist_ok=False)


def check(name, ok, evidence=None, hard=True):
    status = 'pass' if ok else ('fail' if hard else 'differs')
    CHECKS.append({'check': name, 'status': status, 'evidence': json.dumps(jsonable(evidence), ensure_ascii=False, allow_nan=False)[:3000]})
    if not ok and hard:
        raise EvaluationError('check failed: ' + name + ' ' + str(evidence)[:800])


def peak_rss_bytes():
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return int(r if platform.system() == 'Darwin' else r * 1024)


@contextlib.contextmanager
def stage(name):
    t0 = time.time()
    print('[stage]', name, flush=True)
    yield
    DURATIONS[name] = {'seconds': round(time.time() - t0, 2), 'peak_rss_bytes_so_far': peak_rss_bytes()}
    print('  done', name, DURATIONS[name], flush=True)


def put_report(name, obj):
    """Aggregate outputs only (no row-level cells). DataFrame -> CSV, str -> UTF-8 text, otherwise strict JSON."""
    p = PATHS['report_dir'] / name
    data = csv_bytes(obj) if isinstance(obj, pd.DataFrame) else obj.encode('utf-8') if isinstance(obj, str) else json_bytes(obj)
    REPORT_FILES[name] = atomic_write_bytes(p, data)
    return p


In [ ]:
# INPUTS: manager pins, accepted 20/19 manifest membership, executed-vs-source 20 cells, lock consistency; population axes and the aggregate
# segment/date attributes (no 2026 rows retained); locked selection and every consumed array (hash, dtype, shape) BEFORE any value is read.
with stage('inputs'):
    PINS, PINS_SHA = load_pins(BACKEND, cfg)
    INPUTS, DOCS = verify_inputs(BACKEND, cfg, PINS)
    _bad = INPUTS[~INPUTS['ok']]
    check('every manager pin, accepted-20 report/data membership, accepted-19 data membership, lock consistency and executed-vs-source 20 cells verified',
          _bad.empty, {'inputs': len(INPUTS), 'failed': _bad.to_dict('records')})
    I = cfg['inputs']
    DD19 = BACKEND / I['prepared_data_dir']
    AXJ = read_strict_json(DD19 / 'population_axes.json')
    SEGS = pd.read_csv(DD19 / 'universe_segments.csv', dtype={'segment_id': str, 'route_code': str})
    DATES = pd.read_csv(DD19 / 'date_features.csv', dtype={'target_date': str})
    S_AX, D_AX = axes_frame(AXJ, SEGS, DATES, cfg)
    del SEGS, DATES
    _pn_path = BACKEND / I['province_names']
    PROVINCE_NAMES = province_name_map(pd.read_csv(_pn_path, dtype={'name_en': str, 'name_th': str}, usecols=['province_code', 'name_en', 'name_th']))
    INPUTS = pd.concat([INPUTS, pd.DataFrame([{'input': I['province_names'], 'source': 'display names only (not a manager pin; labels, never metrics)',
                                               'sha256': sha256_file(_pn_path), 'expected': None, 'ok': True}])], ignore_index=True)
    _codes = sorted({c for c in province_labels(S_AX) if c != 'unknown'}, key=int)
    check('every known segment province code has a display name (names are labels only; a missing name shows the code)',
          all(c in PROVINCE_NAMES for c in _codes), {'codes': _codes, 'missing': [c for c in _codes if c not in PROVINCE_NAMES]}, hard=False)
    NS = len(AXJ['segments'])
    KM = S_AX['geom_length_m'].to_numpy(np.float64) / 1000.0
    NBLK = {'daily': 1, '6h': len(cfg['time']['blocks_6h'])}
    check('population axes: sorted segments, 6 h blocks as configured, no axis date after ' + cfg['time']['max_date_read'],
          AXJ['segments'] == sorted(AXJ['segments']) and AXJ['blocks'] == cfg['time']['blocks_6h'] and AXJ['dates'][-1] <= cfg['time']['max_date_read'],
          {'segments': NS, 'dates': [AXJ['dates'][0], AXJ['dates'][-1]], 'blocks': AXJ['blocks'], 'network_km': float(KM.sum())})
    SEL = DOCS['selection']['units']
    GATES = DOCS['config20']['selection']
    METHODS = {u: {'selected': SEL[u]['selected_method'], 'baseline': SEL[u]['strongest_baseline']} for u in UNITS}
    check('frozen selection fields are self-consistent (selected = best candidate only when recommended, else the baseline)',
          all(SEL[u]['selected_method'] == (SEL[u]['best_candidate'] if SEL[u]['recommended'] == 'candidate' else SEL[u]['strongest_baseline']) for u in UNITS),
          {u: {k: SEL[u][k] for k in ('recommended', 'selected_method', 'strongest_baseline', 'reason')} for u in UNITS})
    IDX = DOCS['index']['units']
    DATE0, PERIODS, NROWS = {}, {}, {}
    for u in UNITS:
        for scope, pk in (('oof', 'oof_periods'), ('diag', 'diag_periods')):
            P = IDX[u][pk]
            NROWS[(u, scope)] = int(IDX[u][scope]['y']['shape'][0])
            DATE0[(u, scope)] = AXJ['dates'].index(P[0]['start'])
            ok = sum(p['rows'] for p in P) == NROWS[(u, scope)] and all(
                p['rows'] == ((pd.Timestamp(p['end']) - pd.Timestamp(p['start'])).days + 1) * NS * NBLK[u] and
                p['row_offset'] == (AXJ['dates'].index(p['start']) - DATE0[(u, scope)]) * NS * NBLK[u] for p in P)
            PERIODS[(u, scope)] = P
            check(f'{u} {scope}: period offsets/rows reproduce date x segment x block axis order', ok, P)
        check(f'{u}: OOF population equals the locked 2022-23 size', NROWS[(u, 'oof')] == cfg['time']['expected_oof_rows'][u], NROWS[(u, 'oof')])
    PRED, Y, MASK, ARRAY_ROWS = {}, {}, {}, []
    for u in UNITS:
        for scope in ('oof', 'diag'):
            for role, m in METHODS[u].items():
                PRED[(u, scope, role)], metas = load_method(DOCS, u, scope, m)
                ARRAY_ROWS += [dict(mm, role=role, method=m) for mm in metas]
            Y[(u, scope)], ym = load_array(DOCS, u, scope, 'y')
            MASK[(u, scope)], mk = load_array(DOCS, u, scope, 'context_mask')
            ARRAY_ROWS += [dict(ym, role='label', method=''), dict(mk, role='context_mask', method='')]
    ARRAYS = pd.DataFrame(ARRAY_ROWS).drop_duplicates(['unit', 'scope', 'name'])
    check('every consumed array hash == arrays_index == accepted manifest membership; dtype/shape as indexed', True, {'arrays': len(ARRAYS)})
put_report('inputs.csv', INPUTS)
put_report('arrays_used.csv', ARRAYS)
ARRAYS[['unit', 'scope', 'name', 'role', 'method', 'shape']]


In [ ]:
# COLOUR CONTRACTS: per unit, P50/P80/P95 of the selected policy's full 2022-23 OOF density (unweighted cells). Written as strict JSON, re-read and
# re-applied to prove the saved numbers reproduce the bands; calibration bin edges (predicted-count quantiles) per role from the same OOF.
with stage('colour_contracts'):
    CONTRACTS, CAL_EDGES = {}, {}
    sel_sha = PINS['files'][cfg['pins']['repo_prefix'] + I['accepted_root'] + '/' + I['report_dir20'] + '/selection_frozen.json']
    for u in UNITS:
        n = NROWS[(u, 'oof')]
        _, s, _ = cell_index(0, n, NS, NBLK[u])
        for role in ('selected', 'baseline'):
            p = PRED[(u, 'oof', role)](0, n)
            check(f'{u} OOF {role} predictions finite and non-negative', bool(np.isfinite(p).all() and (p >= 0).all()), {'n': int(p.size)})
            CAL_EDGES[(u, role)] = calibration_edges(p, cfg['calibration']['predicted_count_quantile_edges'])
            if role == 'selected':
                dens, ok = segment_density(p, KM[s] * 1000.0)
                check(f'{u}: every reference cell has a valid density', bool(ok.all()), int((~ok).sum()))
                ref = {'method': METHODS[u]['selected'], 'scope': 'OOF ' + cfg['time']['oof_start'] + '..' + cfg['time']['oof_end'] + ' full population',
                       'weighting': 'unweighted cells (preregistered)', 'experiment_lock_sha256': DOCS['selection']['experiment_lock_sha256'],
                       'selection_frozen_sha256': sel_sha, 'source_arrays': ARRAYS[(ARRAYS.unit == u) & (ARRAYS.scope == 'oof') & (ARRAYS.role == 'selected')][['file', 'sha256']].to_dict('records'),
                       'length_source': 'universe_segments.csv geom_length_m (accepted 19)'}
                c = build_colour_contract(dens, u, cfg['colour'], ref)
                put_report(f'colour_contract_{u}.json', c)
                CONTRACTS[u] = read_strict_json(PATHS['report_dir'] / f'colour_contract_{u}.json')
                again = assign_colour_codes(p, KM[s] * 1000.0, CONTRACTS[u], u)
                cnt = np.bincount(again, minlength=GREY_CODE + 1)
                check(f'{u}: saved strict-JSON cutoffs reproduce the reference band counts (no grey in the reference)',
                      cnt[GREY_CODE] == 0 and {COLOUR_LABELS[i]: int(cnt[i]) for i in range(GREY_CODE)} == CONTRACTS[u]['reference_cell_counts'],
                      {'raw_cutoffs': CONTRACTS[u]['raw_cutoffs'], 'counts': cnt.tolist(), 'labels_in_use': CONTRACTS[u]['labels_in_use']})
                del dens, ok, again
            del p
        del s
    other = {'daily': '6h', '6h': 'daily'}
    refused = []
    for u in UNITS:
        try:
            assign_colour_codes(np.array([1.0]), np.array([1000.0]), CONTRACTS[other[u]], u)
        except EvaluationError:
            refused.append(u)
    check('a contract is refused for the other horizon (daily cutoffs never applied to 6 h and vice versa)', refused == list(UNITS), refused)
pd.DataFrame([{'unit': u, 'raw_cutoffs': c['raw_cutoffs'], 'labels_in_use': c['labels_in_use'], **c['reference_cell_counts']} for u, c in CONTRACTS.items()])


In [ ]:
# EVALUATION: one bounded streamed pass per unit x scope (OOF 2022-23, previously seen 2024/25) over the selected policy and the strongest
# baseline: full-population metrics by every group dimension, colour band (locked contract), calibration bin, metric-floor group and
# period x colour (evaluate_scope, FUNCTIONS 2). Then reconciliation with the accepted 20 tables, the locked gates re-derived (confirmation
# only), the floor audit and the province population reconciliation against the input segment codes.
with stage('evaluation'):
    MET, ACCS = [], {}
    for u in UNITS:
        for scope in ('oof', 'diag'):
            t0 = time.time()
            ctx = {'S': S_AX, 'D': D_AX, 'cfg': cfg, 'periods': PERIODS[(u, scope)], 'methods': METHODS[u],
                   'pred': {r: PRED[(u, scope, r)] for r in METHODS[u]}, 'y': Y[(u, scope)], 'mask': MASK[(u, scope)], 'n_rows': NROWS[(u, scope)],
                   'n_seg': NS, 'n_blk': NBLK[u], 'date0': DATE0[(u, scope)], 'km': KM, 'contract': CONTRACTS[u],
                   'cal_edges': {r: CAL_EDGES[(u, r)] for r in METHODS[u]}, 'floor': FLOOR, 'batch_rows': BR}
            df, ACCS[(u, scope)] = evaluate_scope(u, scope, ctx)
            MET.append(df)
            print(f'  {u} {scope}: {NROWS[(u, scope)]} rows in {time.time() - t0:.1f}s', flush=True)
    MET = pd.concat(MET, ignore_index=True)
    rep = DOCS['report_dir20']
    LB = pd.read_csv(rep / 'leaderboard_oof.csv', float_precision='round_trip', dtype={'method': str})
    FM = pd.read_csv(rep / 'fold_metrics.csv', float_precision='round_trip', dtype={'method': str})
    DG = pd.read_csv(rep / 'diagnostics_seen_2024_2025.csv', float_precision='round_trip', dtype={'method': str})
    REC, SELROWS, FLOOR_ROWS, FLOOR_SUM = [], [], [], []
    for u in UNITS:
        for r, m in METHODS[u].items():
            g = lambda scope, dim, grp: MET[(MET.unit == u) & (MET.scope == scope) & (MET.role == r) & (MET.dimension == dim) & (MET.group == grp)].iloc[0].to_dict()
            pooled = g('oof', 'pooled', 'all')
            refs = [('leaderboard_oof.csv', LB[(LB.unit == u) & (LB.method == m)].iloc[0].to_dict(), pooled),
                    ('selection_frozen.' + ('candidate' if r == 'selected' and SEL[u]['recommended'] == 'candidate' else 'baseline') + '_pooled',
                     SEL[u]['candidate_pooled' if r == 'selected' and SEL[u]['recommended'] == 'candidate' else 'baseline_pooled'], pooled)]
            refs += [('fold_metrics.csv ' + p['id'], FM[(FM.unit == u) & (FM.fold_id == p['id']) & (FM.method == m)].iloc[0].to_dict(), g('oof', 'period', p['id']))
                     for p in PERIODS[(u, 'oof')]]
            refs += [('diagnostics_seen_2024_2025.csv ' + p['id'], DG[(DG.unit == u) & (DG.year == p['id']) & (DG.method == m)].iloc[0].to_dict(), g('diag', 'period', p['id']))
                     for p in PERIODS[(u, 'diag')]]
            for name, ref, mine in refs:
                REC.append({'unit': u, 'role': r, 'method': m, 'reference': name, 'problems': '; '.join(reconcile(mine, ref, KEYS, RT, AT))})
        pick = lambda r, scope, dim, grp: MET[(MET.unit == u) & (MET.scope == scope) & (MET.role == r) & (MET.dimension == dim) & (MET.group == grp)].iloc[0].to_dict()
        folds = [p['id'] for p in PERIODS[(u, 'oof')]]
        gs = gate_status(SEL[u], pick('baseline', 'oof', 'pooled', 'all'), pick('selected', 'oof', 'pooled', 'all'),
                         {f: pick('baseline', 'oof', 'period', f) for f in folds}, {f: pick('selected', 'oof', 'period', f) for f in folds}, GATES)
        if SEL[u]['recommended'] != 'candidate':
            gs['note'] = 'baseline retained by the locked gates; selected == baseline, so candidate-vs-baseline gains are not claimed'
        SELROWS.append(dict(unit=u, selected_method=METHODS[u]['selected'], baseline=METHODS[u]['baseline'], best_candidate=SEL[u]['best_candidate'],
                            reason=SEL[u]['reason'], locked_gates=GATES, **gs))
        mref = cfg['manager_reference']
        check(f'{u}: recomputed gate values equal the frozen selection and the manager-accepted figures (confirmation only, no reselection)',
              gs['consistent_with_frozen'] and math.isclose(gs['relative_deviance_reduction'], mref['relative_deviance_reduction'][u], abs_tol=mref['reference_tolerance'])
              and math.isclose(gs['mae_ratio'], mref['mae_ratio'][u], abs_tol=mref['reference_tolerance']), gs)
        for scope in ('oof', 'diag'):
            rows, summ = floor_audit_rows(ACCS[(u, scope)][('baseline', 'floor')], ACCS[(u, scope)][('selected', 'floor')], u, scope)
            FLOOR_ROWS += rows
            FLOOR_SUM.append(summ)
        fs = [s for s in FLOOR_SUM if s['unit'] == u and s['scope'] == 'oof'][0]
        check(f'{u}: floor-group gain share reproduces the manager audit (reported limitation, not a gate)',
              fs['baseline_floor_gain_share'] is not None and abs(fs['baseline_floor_gain_share'] - mref['floor_group_gain_share'][u]) <= mref['reference_tolerance'],
              {'recomputed': fs['baseline_floor_gain_share'], 'manager': mref['floor_group_gain_share'][u]}, hard=False)
        dm = int(np.asarray(MASK[(u, 'diag')]).sum())
        check(f'{u}: 2024/25 routing masks (context available cells) reported; zero means the selected policy used its core component there',
              True, {'context_available_cells_2024_2025': dm, 'cells': NROWS[(u, 'diag')]})
    REC = pd.DataFrame(REC)
    check('recomputed pooled / fold / seen-year metrics reconcile with the accepted 20 tables and the frozen selection', (REC.problems == '').all(),
          REC[REC.problems != ''].to_dict('records'))
    SELROWS = pd.DataFrame(SELROWS)
    FLOOR_ROWS, FLOOR_SUM = pd.DataFrame(FLOOR_ROWS), pd.DataFrame(FLOOR_SUM)
    zero_actual = MET[(MET.total_actual == 0)]
    check('groups with zero actual records carry WAPE = undefined (never 0 or inf)', (zero_actual.wape.isna() & (zero_actual.wape_status == 'undefined_zero_actual')).all(),
          int(len(zero_actual)))
    PROV_REC = [province_reconciliation(S_AX, MET, u, scope, NROWS[(u, scope)]) for u in UNITS for scope in ('oof', 'diag')]
    check('province groups reconcile with the input segment province codes: unknown kept exactly when a code is missing (zero expected '
          'unknown verified explicitly), every known province population present, records sum to pooled (checks accumulation; same label mapping)',
          all(not p['problems'] and p['unknown_group_present'] == (p['expected_unknown_cells'] > 0) for p in PROV_REC), PROV_REC)
put_report('metrics_by_group.csv', MET)
put_report('reconciliation.csv', REC)
put_report('selection_reason.csv', SELROWS.assign(locked_gates=SELROWS.locked_gates.map(lambda v: json.dumps(v, sort_keys=True)),
                                                    fold_wins=SELROWS.fold_wins.map(lambda v: json.dumps(v, sort_keys=True)), gates=SELROWS.gates.map(lambda v: json.dumps(v, sort_keys=True))))
put_report('floor_audit.csv', FLOOR_ROWS)
put_report('floor_audit_summary.csv', FLOOR_SUM)
put_report('province_reconciliation.csv', pd.DataFrame(PROV_REC).assign(roles=lambda d: d.roles.map(','.join), problems=lambda d: d.problems.map('; '.join)))
SELROWS[['unit', 'selected_method', 'baseline', 'relative_deviance_reduction', 'mae_ratio', 'n_fold_wins', 'all_pass', 'baseline_retained']]


In [ ]:
# CAPTURE + STABILITY + TRANSFER: equal-road-km capture per day / 6 h block for each role's own density ranking (selected vs baseline vs the
# uniform road-km expectation), colour exposure / record capture with the SAME locked cutoffs for OOF, each fold and each seen year, the
# spatial-transfer metrics recomputed and kept separate from temporal selection, and the notebook-20 exploratory bootstrap (not recomputed).
FR = [float(f) for f in cfg['capture']['road_km_fractions']]


def capture_rows(u, scope):
    n, nb = NROWS[(u, scope)], NBLK[u]
    y = to_period_matrix(np.asarray(Y[(u, scope)][:n], np.float64), NS, nb)
    dates = np.arange(y.shape[0]) // nb + DATE0[(u, scope)]
    blocks = np.arange(y.shape[0]) % nb
    groups = [('oof_2022_2023' if scope == 'oof' else 'seen_2024_2025', np.ones(y.shape[0], bool))]
    for p in PERIODS[(u, scope)]:
        a, b = AXJ['dates'].index(p['start']), AXJ['dates'].index(p['end'])
        groups.append((p['id'], (dates >= a) & (dates <= b)))
    out = []
    for r in METHODS[u]:
        p = to_period_matrix(PRED[(u, scope, r)](0, n), NS, nb)
        dens = p / KM[None, :]
        res, ty, tp = capture_by_period(dens, y, p, KM, FR)
        for gname, gm in groups:
            for bi, blab in [(-1, 'all')] + ([(i, l) for i, l in enumerate(cfg['time']['blocks_6h'])] if nb > 1 else []):
                m = gm if bi < 0 else gm & (blocks == bi)
                tot, totp = float(ty[m].sum()), float(tp[m].sum())
                has = m & (ty > 0)
                for f in FR:
                    ca, cp = float(res[f]['actual'][m].sum()), float(res[f]['pred'][m].sum())
                    out.append({'unit': u, 'scope': gname, 'time_block': blab, 'role': r, 'method': METHODS[u][r], 'road_km_fraction': f,
                                'periods': int(m.sum()), 'periods_with_records': int(has.sum()), 'total_records': tot, 'captured_records': ca,
                                'capture_share': ca / tot if tot > 0 else None, 'uniform_expectation': f, 'lift_vs_uniform': ca / tot / f if tot > 0 else None,
                                'predicted_share': cp / totp if totp > 0 else None,
                                'mean_period_capture_share': float(np.mean(res[f]['actual'][has] / ty[has])) if has.any() else None})
        del p, dens, res
    return out


def colour_rows(u, scope):
    labs = list(COLOUR_LABELS) + [GREY]
    out = []
    for dim, split in (('colour', None), ('period_colour', '|')):
        a = ACCS[(u, scope)][('selected', dim)]
        names = labs if split is None else [p['id'] + '|' + c for p in PERIODS[(u, scope)] for c in labs]
        F = {k: a[:, FIELDS.index(k)] for k in ('n', 'km', 'actual', 'pred')}
        per = np.zeros(len(names), np.int64) if split is None else np.repeat(np.arange(len(PERIODS[(u, scope)])), len(labs))
        for i, nm in enumerate(names):
            pid, lab = ('oof_2022_2023' if scope == 'oof' else 'seen_2024_2025', nm) if split is None else nm.split('|')
            tot = {k: F[k][per == per[i]].sum() for k in F}
            out.append({'unit': u, 'scope': pid, 'colour': lab, 'cells': int(F['n'][i]), 'cell_share': F['n'][i] / tot['n'],
                        'exposure_segment_km_periods': F['km'][i], 'exposure_share': F['km'][i] / tot['km'],
                        'records': F['actual'][i], 'record_share': F['actual'][i] / tot['actual'] if tot['actual'] > 0 else None,
                        'predicted_records': F['pred'][i], 'observed_records_per_km_period': F['actual'][i] / F['km'][i] if F['km'][i] > 0 else None,
                        'predicted_records_per_km_period': F['pred'][i] / F['km'][i] if F['km'][i] > 0 else None})
    return out


with stage('capture_stability_transfer'):
    CAP, COL = [], []
    for u in UNITS:
        for scope in ('oof', 'diag'):
            CAP += capture_rows(u, scope)
            COL += colour_rows(u, scope)
    CAP, COL = pd.DataFrame(CAP), pd.DataFrame(COL)
    tot_check = CAP[(CAP.time_block == 'all') & (CAP.road_km_fraction == FR[0])].groupby(['unit', 'scope', 'role']).total_records.first()
    check('capture totals equal the label totals of every scope (no record lost at period boundaries)',
          all(math.isclose(tot_check[(u, 'oof_2022_2023', r)], MET[(MET.unit == u) & (MET.scope == 'oof') & (MET.role == r) & (MET.dimension == 'pooled')].total_actual.iloc[0])
              for u in UNITS for r in METHODS[u]), tot_check.to_dict())
    mono = {}
    for (u, sc), d in COL[COL.cells > 0].groupby(['unit', 'scope']):
        d = d[d.colour != GREY].set_index('colour').reindex([c for c in COLOUR_LABELS if c in set(d.colour)])
        mono[u + '|' + sc] = bool(np.all(np.diff(d.observed_records_per_km_period.to_numpy(float)) >= 0))
    check('observed density is non-decreasing across in-use colour bands (reported per scope; a reversal is disclosed, never re-fitted)', all(mono.values()), mono, hard=False)
    TR = []
    trep = pd.read_csv(DOCS['report_dir20'] / 'transfer_metrics.csv', float_precision='round_trip', dtype={'method': str})
    for u in UNITS:
        ty, tym = load_array(DOCS, u, 'transfer', 'y')
        yv = np.asarray(ty, np.float64)
        names = [k for k in IDX[u]['transfer'] if k != 'y']
        for m in names:
            a, am = load_array(DOCS, u, 'transfer', m)
            acc = new_acc(1)
            acc_add(acc, yv, np.asarray(a, np.float64), np.zeros(yv.size), FLOOR, np.zeros(yv.size, np.int64))
            mm = acc_metrics(acc[0])
            ref = trep[(trep.unit == u) & (trep.method == m)].iloc[0].to_dict()
            TR.append(dict(unit=u, method=m, kind=ref.get('kind'), problems='; '.join(reconcile(mm, ref, KEYS, RT, AT)),
                           **{k: mm[k] for k in ('n', 'mean_poisson_deviance', 'mae', 'rmse', 'bias', 'total_pred', 'total_actual', 'pred_over_actual')}))
    TR = pd.DataFrame(TR)
    base_best = TR[~TR.method.str.contains('::')].groupby('unit').mean_poisson_deviance.min()
    TR['relative_to_best_baseline'] = [1.0 - r.mean_poisson_deviance / base_best[r.unit] for r in TR.itertuples()]
    TR['note'] = ['candidate core config (selected routed policy is not transfer-evaluated)' if '::' in m else 'baseline with explicit unseen-segment fallback' for m in TR.method]
    check('spatial-transfer metrics recomputed from the transfer arrays equal transfer_metrics.csv', (TR.problems == '').all(), TR[TR.problems != ''].to_dict('records'))
    BOOT = DOCS['selection']['bootstrap_exploratory']
put_report('capture_equal_road_km.csv', CAP)
put_report('colour_exposure_capture.csv', COL)
put_report('spatial_transfer.csv', TR)
put_report('bootstrap_exploratory.json', {'source': 'notebook 20 selection_frozen.json (paired day-block bootstrap; exploratory, post-selection)', 'units': BOOT})
CAP[(CAP.scope == 'oof_2022_2023') & (CAP.time_block == 'all')][['unit', 'role', 'road_km_fraction', 'capture_share', 'lift_vs_uniform']]


In [ ]:
# FIGURES + REPORTS + MANIFEST: every chart is drawn from an aggregate CSV written in this run (figure_*.csv); Thai results, Model Gate
# narrative and README; protected paths re-hashed; strict-JSON manifest with sources, config, inputs, outputs, environment, durations, checks.
with stage('figures_reports'):
    FIGS = []
    figdir = PATHS['report_dir'] / cfg['outputs']['figure_subdir']
    bl = {u: LB[(LB.unit == u) & (LB.method == METHODS[u]['baseline'])].mean_poisson_deviance.iloc[0] for u in UNITS}
    FMC = LB[LB.method.str.contains('::') | LB.method.isin([METHODS[u]['baseline'] for u in UNITS])].copy()
    FMC['relative_deviance_reduction_vs_strongest_baseline'] = [1.0 - r.mean_poisson_deviance / bl[r.unit] for r in FMC.itertuples()]
    FMC = FMC[['unit', 'method', 'kind', 'deployment_eligible', 'n', 'mean_poisson_deviance', 'mae', 'rmse', 'relative_deviance_reduction_vs_strongest_baseline']]
    put_report('figure_method_comparison.csv', FMC)
    FIGS += save_figure(fig_bars(FMC, UNITS, 'method', 'relative_deviance_reduction_vs_strongest_baseline',
                                 'Pooled 2022-23 OOF mean Poisson deviance: % reduction vs strongest baseline (segment_mean_k365)',
                                 '% deviance reduction (positive = lower deviance than baseline)', pct=True, hline=GATES['min_relative_deviance_reduction'],
                                 horizontal=True, colour_by='kind',
                                 note='dashed = locked 1% gate; every bar uses all OOF cells of its unit (daily 2022-23 segment-days, 6 h segment-blocks); '
                                      'context = ablation, never eligible; source figure_method_comparison.csv'), figdir, 'method_comparison')
    P = MET[(MET.dimension == 'period')]
    piv = P.pivot_table(index=['unit', 'scope', 'group'], columns='role', values='mean_poisson_deviance').reset_index()
    piv['relative_deviance_reduction'] = 1.0 - piv['selected'] / piv['baseline']
    piv = piv.rename(columns={'selected': 'selected_mean_deviance', 'baseline': 'baseline_mean_deviance'})
    put_report('figure_period_performance.csv', piv)
    FIGS += save_figure(fig_bars(piv[piv.scope == 'oof'], UNITS, 'group', 'relative_deviance_reduction', 'Per-fold OOF mean Poisson deviance reduction, selected vs baseline',
                                 '% deviance reduction (positive = lower)', pct=True,
                                 note='each fold = all segment x period cells of that half-year; source figure_period_performance.csv'), figdir, 'fold_performance')
    FIGS += save_figure(fig_bars(piv[piv.scope == 'diag'], UNITS, 'group', 'relative_deviance_reduction',
                                 'Previously seen 2024 / 2025: mean Poisson deviance reduction, selected vs baseline (diagnostic only, never used to select)',
                                 '% deviance reduction (positive = lower)', pct=True,
                                 note='not an untouched test; 2024/25 context-available cells (routed policy uses its core fallback elsewhere): '
                                      + ', '.join(f'{u} {int(np.asarray(MASK[(u, "diag")]).sum()):,}' for u in UNITS)
                                      + '; 2026 pending a separate audit; source figure_period_performance.csv'), figdir, 'seen_2024_2025')
    CAL = MET[(MET.scope == 'oof') & (MET.dimension == 'calibration') & (MET.n > 0)][['unit', 'role', 'method', 'group', 'n', 'total_pred', 'total_actual',
                                                                                        'exposure_segment_km_periods', 'predicted_records_per_km_period', 'observed_records_per_km_period']]
    put_report('figure_calibration.csv', CAL)
    FIGS += save_figure(fig_calibration(CAL, UNITS), figdir, 'calibration')
    GDIMS = ['actual_zero', 'context', 'time_block', 'season', 'weekday', 'festival']
    ADIMS = ['province', 'month']
    gp = group_gain_table(MET, 'oof', GDIMS + ADIMS)
    gp['label'] = [group_label(k, g, PROVINCE_NAMES) for k, g in zip(gp.dimension, gp.group)]
    put_report('figure_errors_by_group.csv', gp)
    FIGS += save_figure(fig_group_panels(gp, UNITS, GDIMS, '2022-23 OOF mean Poisson deviance reduction by group, selected vs baseline (descriptive, post-selection)',
                                         note='n = segment x period cells, rec = selected+linked records in the group; red = baseline lower deviance; '
                                              'no causes inferred from predictors; source figure_errors_by_group.csv', names=PROVINCE_NAMES), figdir, 'errors_by_group')
    FIGS += save_figure(fig_group_panels(gp, UNITS, ADIMS, '2022-23 OOF mean Poisson deviance reduction by province and month, selected vs baseline (descriptive)',
                                         note='province = segment-midpoint province on the current geometry (code + name from province_points.csv), not event attribution; '
                                              'n = cells, rec = records; small supports are noisy; source figure_errors_by_group.csv', names=PROVINCE_NAMES), figdir, 'errors_by_area_month')
    CL = COL[COL.scope.isin(['oof_2022_2023'])].melt(id_vars=['unit', 'colour'], value_vars=['cell_share', 'exposure_share', 'record_share'], var_name='measure', value_name='share')
    put_report('figure_colour_exposure_capture.csv', CL)
    FIGS += save_figure(fig_bars(CL, UNITS, 'colour', 'share', 'Locked colour bands, 2022-23 OOF: share of cells, segment-km-periods and records',
                                 '% of total', hue='measure', pct=True,
                                 note='denominators = all OOF cells / total segment-km-periods / total selected+linked records of the unit; '
                                      'base = lower relative density, NOT safe; source figure_colour_exposure_capture.csv'), figdir, 'colour_exposure_capture')
    FIGS += save_figure(fig_capture(CAP, UNITS), figdir, 'capture_curve')
    FL = FLOOR_ROWS[FLOOR_ROWS.scope == 'oof']
    put_report('figure_floor_audit.csv', FL)
    FIGS += save_figure(fig_bars(FL, UNITS, 'floor_group', 'share_of_total_gain', 'Share of pooled 2022-23 OOF deviance gain by metric-floor group (post-hoc sensitivity, not a gate)',
                                 '% of total deviance gain', pct=True,
                                 note=f'floor groups: whether baseline / candidate predict < {FLOOR!r} (scored at the floor); cells and positive cells per group in '
                                      'figure_floor_audit.csv'), figdir, 'floor_audit')
    put_report('figure_spatial_transfer.csv', TR)
    FIGS += save_figure(fig_bars(TR, UNITS, 'method', 'relative_to_best_baseline', 'Spatial transfer 2023: held-out segments, separate from temporal selection',
                                 '% deviance reduction vs best transfer baseline', pct=True, horizontal=True,
                                 note='candidate core configs only (the selected routed policy is not transfer-evaluated); one held-out area; '
                                      'source figure_spatial_transfer.csv'), figdir, 'spatial_transfer')
    diag_tab = MET[(MET.scope == 'diag') & (MET.dimension == 'period')][['unit', 'group', 'role', 'mean_poisson_deviance', 'mae', 'pred_over_actual']]
    sel_list = SELROWS.to_dict('records')
    put_report('results_th.md', results_th(sel_list, FLOOR_SUM.to_dict('records'), CONTRACTS, CAP, diag_tab,
                                           TR[['unit', 'method', 'mean_poisson_deviance', 'relative_to_best_baseline']], BOOT))
    put_report('model_gate_presentation_th.md', presentation_th({
        'label_meaning': DOCS['manifest20'].get('label_meaning') or DOCS['lock'].get('label_meaning'), 'sel_rows': sel_list, 'config20': DOCS['config20'],
        'feature_columns': DOCS['lock']['feature_columns'], 'leaderboard': FMC, 'leaderboard_all': LB, 'met': MET,
        'floor_rows': FLOOR_ROWS[FLOOR_ROWS.scope == 'oof'], 'floor_sum': FLOOR_SUM[FLOOR_SUM.scope == 'oof'].to_dict('records'), 'cap': CAP,
        'colour_rows': COL, 'transfer': TR, 'boot': BOOT, 'contracts': CONTRACTS, 'figs': FIGS, 'province_names': PROVINCE_NAMES,
        'network': {'segments': NS, 'network_km': float(KM.sum())}, 'capture_cfg': cfg['capture'],
        'diag_context_cells': {u: int(np.asarray(MASK[(u, 'diag')]).sum()) for u in UNITS}}))
    _heads = [l[3:] for l in (PATHS['report_dir'] / 'model_gate_presentation_th.md').read_text('utf-8').splitlines() if l.startswith('## ')]
    check('Model Gate presentation has exactly the five required headings in order', _heads == list(GATE_HEADINGS), _heads)
    put_report('README_th.md', '\n'.join([
        '# spatial_evaluation (notebook 21) -- คำอธิบายไฟล์', '',
        '- colour_contract_daily.json / colour_contract_6h.json: เกณฑ์สีคงที่ (P50/P80/P95 ของ density OOF 2022-23 ของ policy ที่เลือก) แยกตามหน่วยเวลา ห้ามแปลงข้ามหน่วย',
        '- metrics_by_group.csv: metric เต็มประชากรตามกลุ่ม (aggregate เท่านั้น ไม่มีข้อมูลรายแถว); WAPE = ว่าง เมื่อกลุ่มไม่มีบันทึกจริง',
        '- reconciliation.csv / selection_reason.csv: เทียบกับตารางของ notebook 20 และค่า gate ที่ล็อกไว้ (ยืนยันเท่านั้น ไม่เลือกใหม่)',
        '- floor_audit*.csv: sensitivity หลังเห็นผล daily เบื้องต้น (ไม่ใช่ gate)',
        '- province_reconciliation.csv: จำนวน cells ต่อจังหวัดเทียบกับ province_code ของ segment ที่เป็น input (unknown แยกกลุ่มเมื่อไม่มี code; ใช้ mapping เดียวกัน จึงไม่ใช่การตรวจอิสระ)',
        '- figure_errors_by_group.csv: ใช้กับ errors_by_group (เวลา/เทศกาล/context/zero) และ errors_by_area_month (จังหวัด/เดือน) พร้อม cells/records ของแต่ละกลุ่ม',
        '- capture_equal_road_km.csv / colour_exposure_capture.csv: capture ที่งบ road-km เท่ากันต่อวัน/ต่อ 6 ชม. และสัดส่วนตามสี (ไม่ใช่ผลด้านความปลอดภัยเชิงสาเหตุ)',
        '- spatial_transfer.csv: ผล transfer แยกจากการเลือกเชิงเวลา; bootstrap_exploratory.json: จาก notebook 20 (exploratory หลังการเลือก)',
        '- figures/*.png|svg + figure_*.csv: กราฟจากตัวเลขจริงในไฟล์ CSV เดียวกัน; results_th.md, model_gate_presentation_th.md: สรุปภาษาไทย',
        '- 2026: รอการตรวจข้อมูลและประเมินแยกต่างหาก -- ยังไม่มีผล', '']))
    PROTECTED_AFTER = hash_paths(BACKEND, cfg['protected_paths'])
    changed = sorted(k for k in set(PROTECTED_BEFORE) | set(PROTECTED_AFTER) if PROTECTED_BEFORE.get(k) != PROTECTED_AFTER.get(k))
    check('protected accepted-20/19, catalog, evaluation pins, configs, notebooks and scripts unchanged', not changed, {'files': len(PROTECTED_BEFORE), 'changed': changed[:50]})
    check('source config unchanged during the run', sha256_file(CONFIG_PATH) == CFG_SHA)
    check('at least eight actual-result figures written as PNG + SVG', len(FIGS) >= 16, FIGS)
    nb_src = BACKEND / cfg['notebook']
    fsrc = [s for s, _ in code_sources(nb_src) if s.lstrip().startswith('# FUNCTIONS')] if nb_src.is_file() else []
MANIFEST = {
    'status': 'complete' if all(c['status'] != 'fail' for c in CHECKS) else 'failed', 'version': cfg['version'],
    'purpose': cfg['purpose'], 'label_meaning': DOCS['manifest20'].get('label_meaning'),
    'started_utc': STARTED, 'finished_utc': utcnow(), 'seconds': round(time.time() - T_START, 2), 'durations': DURATIONS,
    'config_sha256': CFG_SHA, 'pins_sha256': PINS_SHA, 'notebook': cfg['notebook'],
    'notebook_sha256': sha256_file(nb_src) if nb_src.is_file() else None,
    'functions_sha256': sha256_bytes(''.join(fsrc).encode('utf-8')) if fsrc else None, 'functions_cells': len(fsrc),
    'accepted_root': I['accepted_root'], 'experiment_lock_sha256': DOCS['selection']['experiment_lock_sha256'],
    'selected': {u: METHODS[u] for u in UNITS}, 'colour_contract_sha256': {u: REPORT_FILES[f'colour_contract_{u}.json'] for u in UNITS},
    'environment': environment(VERSIONS), 'inputs': INPUTS.to_dict('records'), 'arrays': ARRAYS.to_dict('records'),
    'checks': CHECKS, 'failed_checks': [c for c in CHECKS if c['status'] == 'fail'], 'differs_checks': [c for c in CHECKS if c['status'] == 'differs'],
    'report_files': dict(REPORT_FILES), 'data_files': dict(DATA_FILES),
    'protected_files_hashed': len(PROTECTED_BEFORE), 'protected_unchanged': not changed, 'protected_sha256_before': PROTECTED_BEFORE,
    'readiness': {'evaluation_complete_for_2022_2025_seen_data': True, 'colour_contract_ready_for_notebook22': True,
                  'serving_recommendation': 'benchmark-only serving of the existing notebook-20 bundles until a separate user need arises; no refit here',
                  '2026': 'pending separate data audit and evaluation; no 2026 data read', 'backend_complete': False},
}
atomic_write_bytes(PATHS['report_dir'] / 'spatial_evaluation_manifest.json', json_bytes(MANIFEST))
print(MANIFEST['status'], len(CHECKS), 'checks;', len(REPORT_FILES), 'report files')
pd.DataFrame(CHECKS)[['check', 'status']]


## Outputs (under the NEW run root only)

`reports/v3/spatial_evaluation/`: `spatial_evaluation_manifest.json`, `colour_contract_{daily,6h}.json`, `metrics_by_group.csv`,
`reconciliation.csv`, `selection_reason.csv`, `floor_audit*.csv`, `capture_equal_road_km.csv`, `colour_exposure_capture.csv`,
`spatial_transfer.csv`, `bootstrap_exploratory.json`, `figure_*.csv` + `figures/*.png|svg`, `results_th.md`, `model_gate_presentation_th.md`, `README_th.md`.

Aggregates only — no row-level outcomes or errors are exported. Independent check: `backend/scripts/verify_spatial_evaluation.py`.
Mirror to `output/spatial-model-results/` only after Codex verification, without replacing older V2 assets.
